In [ ]:
import os
import threading
import queue
import time
import re
import sounddevice as sd # For audio I/O, PortAudioError
import numpy as np
from PIL import Image
import pytesseract

# Your existing utils
from utils.model import initiate_llm, initiate_tts_model
from utils.to_base64 import encode_image_to_base64
from utils.audio_player import audio_player
from utils.audio_generator import audio_generator
from utils.snap_a_picture import capture_image

# LangChain imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

# --- Placeholder for STT ---
def listen_for_command():
    """Placeholder: Implement Speech-to-Text here."""
    print("Listening for voice command...")
    # In a real implementation, this would use whisper, vosk, etc.
    # and return the transcribed text.
    time.sleep(2) # Simulate listening
    # command = input("Enter command (or 'quit'): ") # Manual input for testing
    # Let's simulate commands for flow testing:
    # command = "describe what's in front of me"
    command = "read this document"
    # command = "how do i get to the elevator"
    # command = "quit"
    print(f"Heard: '{command}'")
    return command

# --- Placeholder for Positioning ---
def get_current_location():
    """Placeholder: Implement indoor positioning system integration."""
    # In reality, get data from VIO/SLAM/Markers
    print("DEBUG: Assuming current location is 'Entrance'")
    return "Entrance" # Example node ID

class Assistant:
    def __init__(self):
        print("Initializing Assistant...")
        # --- Config ---
        self.lm_studio_base_url = "http://localhost:1234/v1"
        # IMPORTANT: Set correct model identifiers
        self.tool_llm_model_id = "hugging-quants/Llama-3.2-3B-Instruct-Q8_0-GGUF/llama-3.2-3b-instruct-q8_0.gguf" # For LangChain Agent
        self.vision_llm_model_id = "lmstudio-community/granite-vision-3.2-2b-GGUF" # For Scene Description
        os.environ["OPENAI_API_KEY"] = "lm-studio" # Dummy key
        self.tts_voice = 'am_onyx'
        self.audio_sample_rate = 24000

        # --- Load Models ---
        print("Initializing TTS...")
        self.tts_pipeline = initiate_tts_model(desired_device='cuda')
        if not self.tts_pipeline:
            raise RuntimeError("Failed to initialize TTS pipeline. Exiting.")

        print("Initializing Tool-Using LLM...")
        # Note: LangChain's ChatOpenAI might not easily handle switching models per call
        # We might need separate instances or direct API calls for the vision model.
        # For now, we initialize the agent LLM here.
        self.agent_llm = ChatOpenAI(
            base_url=self.lm_studio_base_url,
            model=self.tool_llm_model_id,
            temperature=0.1,
        )
        # Vision LLM will be initialized *within* the scene description function if needed

        # --- Define Tools (linking methods of this class) ---
        @tool
        def scene_description_tool() -> str:
            """
            Captures an image of the current surroundings and provides a detailed verbal description focusing on objects, layout, and potential obstacles for a blind user.
            Use this when the user asks 'what is around me?', 'describe the scene', 'what do you see?', etc.
            """
            return self._execute_scene_description()

        @tool
        def ocr_text_tool() -> str:
            """
            Captures an image, extracts any readable text using OCR, and reads it aloud.
            Use this when the user asks 'read this sign', 'what does this document say?', 'read the text'.
            """
            return self._execute_ocr()

        @tool
        def indoor_navigation_tool(destination: str) -> str:
            """
            Provides turn-by-turn indoor navigation directions from the user's current estimated location to a known destination within the building.
            Requires the destination name (e.g., 'reception', 'elevator', 'Room 101').
            """
            # Assumes self.floor_plan exists
            current_loc = get_current_location() # Get live location
            return self._execute_indoor_navigation(current_loc, destination)

        self.tools = [scene_description_tool, ocr_text_tool, indoor_navigation_tool]
        self.llm_with_tools = self.agent_llm.bind_tools(self.tools)
        self.chat_history = [SystemMessage(content="You are a helpful assistant for blind users. Use the available tools proactively to answer questions about surroundings, read text, or provide navigation. Be concise but informative.")]

        # --- Load Map Data ---
        self.floor_plan = { # Example, load from JSON in reality
            "Entrance": {"path": "Hallway A, then turn left at the water cooler to reach Room 101"},
            "Room 101": {"path": "Hallway A, then turn right towards the main doors to reach Entrance"},
            "Elevator": {"path": "Hallway B, past the reception desk"},
        }
        print("Assistant Initialized.")

    def speak(self, text: str):
        """Uses the shared TTS pipeline to speak text."""
        if not text or not self.tts_pipeline:
            return
        print(f"Speaking: {text}")
        # Simple, non-streaming speaking for agent responses/tool results
        # This reuses your generator/player but in a blocking way for simplicity here
        # A dedicated simple speak function would be better.
        temp_audio_queue = queue.Queue()
        temp_text_queue = queue.Queue()
        temp_gen_done = threading.Event()
        temp_text_ready = threading.Event()
        temp_audio_playing = threading.Event()

        temp_text_queue.put(text)
        temp_gen_done.set() # Signal no more text coming

        # This setup mimics the streaming one but just for one chunk
        gen_thread = threading.Thread(target=audio_generator, args=(temp_text_queue, temp_audio_queue, temp_gen_done, temp_text_ready, self.tts_pipeline, self.tts_voice, 1), daemon=True)
        play_thread = threading.Thread(target=audio_player, args=(temp_audio_queue, temp_gen_done, temp_text_ready, temp_audio_playing, self.audio_sample_rate, 0.05, 0.1, 0.1), daemon=True)

        gen_thread.start()
        play_thread.start()
        gen_thread.join()
        play_thread.join()
        print("Speaking finished.")


    def _execute_scene_description(self) -> str:
        """Internal: Handles the logic for the scene description tool."""
        self.speak("Okay, looking around...")
        try:
            # Assuming sceene_description_with_tts is modified to:
            # 1. Accept tts_pipeline instance
            # 2. Initialize its own vision LLM using self.vision_llm_model_id
            # 3. Perform streaming TTS
            # 4. Return the full description string
            full_description = sceene_description_with_tts_modified(self.tts_pipeline, self.vision_llm_model_id, self.lm_studio_base_url)
            # No need to speak here, sceene_description_with_tts_modified does it.
            return full_description # Return text for agent context
        except Exception as e:
            error_msg = f"Sorry, I encountered an error describing the scene: {e}"
            print(error_msg)
            self.speak(error_msg)
            return error_msg

    def _execute_ocr(self) -> str:
        """Internal: Handles the logic for the OCR tool."""
        self.speak("Okay, trying to read...")
        try:
            image_path = capture_image()
            if not image_path or not os.path.exists(image_path):
                raise FileNotFoundError("Failed to capture or find image.")
            image = Image.open(image_path)
            # Consider adding image preprocessing here (grayscale, thresholding)
            text = pytesseract.image_to_string(image)
            result_text = text.strip() if text.strip() else "No readable text found."
            self.speak(result_text) # Speak the result
            return result_text # Return text for agent context
        except Exception as e:
            error_msg = f"Sorry, I encountered an error reading the text: {e}"
            print(error_msg)
            self.speak(error_msg)
            return error_msg

    def _execute_indoor_navigation(self, current_location: str, destination: str) -> str:
        """Internal: Handles the logic for indoor navigation."""
        self.speak(f"Okay, navigating from {current_location} to {destination}.")
        if current_location in self.floor_plan and destination in self.floor_plan:
            # Basic path lookup - real system needs graph search (A*)
            # and integration with continuous positioning + turn detection
            path_description = f"From {current_location}, follow {self.floor_plan[current_location].get('path', 'the marked path')} towards {destination}."
            self.speak(path_description)
            return path_description # Return text for agent context
        else:
            error_msg = f"Sorry, I don't have navigation information for {current_location} or {destination}."
            self.speak(error_msg)
            return error_msg

    def run(self):
        """Main interaction loop."""
        self.speak("Assistant ready. How can I help?")
        while True:
            try:
                command = listen_for_command()
                if not command:
                    continue
                if command.lower() == 'quit':
                    self.speak("Goodbye!")
                    break

                # Add command to history
                self.chat_history.append(HumanMessage(content=command))

                # --- Invoke Agent ---
                print("Invoking agent...")
                ai_response = self.llm_with_tools.invoke(self.chat_history)
                self.chat_history.append(ai_response) # Add raw response first

                final_response_to_speak = ""

                # --- Process Tool Calls ---
                if ai_response.tool_calls:
                    print(f"Agent wants to call tools: {ai_response.tool_calls}")
                    # Execute tools and gather results
                    tool_outputs = []
                    for tool_call in ai_response.tool_calls:
                        tool_name = tool_call['name']
                        tool_args = tool_call['args']
                        tool_id = tool_call['id']

                        # Find the matching tool function object
                        selected_tool = next((t for t in self.tools if t.name == tool_name), None)

                        if selected_tool:
                            try:
                                # Execute the tool's *actual* implementation method
                                # (which might call _execute_scene_description, etc.)
                                # The tool decorator handles calling the right function
                                tool_result = selected_tool.invoke(tool_args)
                                tool_outputs.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))
                            except Exception as e:
                                print(f"Error executing tool {tool_name}: {e}")
                                tool_outputs.append(ToolMessage(content=f"Error executing tool {tool_name}: {e}", tool_call_id=tool_id))
                        else:
                             tool_outputs.append(ToolMessage(content=f"Error: Tool '{tool_name}' not found.", tool_call_id=tool_id))

                    # Add tool results to history and potentially get a final summary from LLM
                    self.chat_history.extend(tool_outputs)
                    print("Sending tool results back to agent for final response...")
                    final_ai_response = self.llm_with_tools.invoke(self.chat_history)
                    self.chat_history.append(final_ai_response)
                    final_response_to_speak = final_ai_response.content

                else:
                    # No tool call, just use the direct content
                    final_response_to_speak = ai_response.content

                # --- Speak Final Response ---
                if final_response_to_speak:
                     # Avoid re-speaking if tool already spoke comprehensively (e.g., scene desc)
                     # This logic needs refinement - maybe tools should return None if they handle speech?
                    if not (ai_response.tool_calls and any(tc['name'] == 'scene_description_tool' for tc in ai_response.tool_calls)):
                         self.speak(final_response_to_speak)
                else:
                    # Handle cases where the LLM might not respond after tool use
                    self.speak("Okay, done.")

            except Exception as e:
                print(f"An error occurred in the main loop: {e}")
                self.speak("Sorry, something went wrong.")
                # Optionally add more robust error recovery here
            finally:
                 # Clean up chat history periodically?
                 if len(self.chat_history) > 10: # Keep last N interactions
                    self.chat_history = self.chat_history[:1] + self.chat_history[-9:]


# --- Need to modify sceene_description_with_tts ---
def sceene_description_with_tts_modified(tts_pipeline, vision_model_id, base_url):
    """
    Modified version for integration.
    - Takes TTS pipeline, vision model ID, base URL.
    - Initializes vision LLM internally.
    - Performs streaming description + TTS.
    - Returns the full text description.
    """
    image_path = capture_image()
    print(f"--- Starting Scene Description with TTS for: {image_path} ---")
    if tts_pipeline is None:
        print("Error: TTS pipeline required.")
        return "Error: TTS pipeline missing."
    if not image_path or not os.path.exists(image_path):
         print("Error: Image capture failed.")
         return "Error: Failed to get image."

    # --- Initialize Queues/Events PER RUN ---
    audio_queue = queue.Queue()
    text_queue = queue.Queue()
    text_ready = threading.Event()
    generation_done = threading.Event()
    audio_playing = threading.Event()
    full_response_text = "" # To store the full text

    # 1. Initialize Vision LLM (inside this function now)
    print(f"Initializing Vision LLM ({vision_model_id})...")
    try:
        # Assuming initiate_llm can handle different base_urls/models
        llm = initiate_llm(base_url, vision_model_id, temperature=0.5)
        print("Vision LLM initialized.")
    except Exception as e:
        print(f"Error initializing Vision LLM: {e}")
        return f"Error initializing vision model: {e}"

    # 2. Encode Image
    base64_image_data_uri = encode_image_to_base64(image_path)
    if not base64_image_data_uri: return "Error encoding image."

    # 3. Prepare Prompt and Message
    prompt = "Describe this image in detail for a person who is blind. Focus on object positions, types, and potential obstacles or points of interest. Be descriptive."
    message = HumanMessage(content=[{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": base64_image_data_uri}}])

    # 4. Start Audio Threads (Pass the tts_pipeline instance)
    audio_player_thread = threading.Thread(target=audio_player, args=(audio_queue, generation_done, text_ready, audio_playing, 24000, 0.05, 0.1, 0.1), daemon=True)
    audio_generator_thread = threading.Thread(target=audio_generator, args=(text_queue, audio_queue, generation_done, text_ready, tts_pipeline, 'am_onyx', 3), daemon=True)
    audio_generator_thread.start()
    audio_player_thread.start()

    # 5. Process LLM Stream and Chunk Text
    print("\nProcessing Vision LLM stream...")
    # (Keep the exact same text chunking logic from your original sceene_description_with_tts)
    sentence_pattern = re.compile(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<![A-Z]\.)(?<=[.?!])\s+|\n\s*')
    accumulated_text = ""

    try:
        for chunk in llm.stream([message]):
            if chunk.content:
                print(chunk.content, end="", flush=True) # Print stream to console
                content = chunk.content
                accumulated_text += content
                full_response_text += content # Append to full response

                # --- Text Chunking Logic (identical to original) ---
                parts = sentence_pattern.split(accumulated_text)
                if len(parts) > 1:
                     for i in range(len(parts) - 1):
                         # ... (find punctuation logic) ...
                         sentence_chunk = parts[i].strip() # + punctuation
                         if sentence_chunk and len(sentence_chunk.split()) >= 3:
                             text_queue.put(sentence_chunk)
                     accumulated_text = parts[-1]
                # ... (MAX_BUFFER_LENGTH logic) ...

        # After loop, process remaining text
        final_chunk = accumulated_text.strip()
        if final_chunk and len(final_chunk.split()) >= 3:
            text_queue.put(final_chunk)

    except Exception as e:
        print(f"\nError during Vision LLM streaming: {e}")
        # We still need to wait for audio to finish what it has
    finally:
        # 6. Signal Generation End and Wait for Audio
        print("\nVision LLM stream finished.")
        generation_done.set()
        print("Waiting for scene description audio...")
        # (Keep the exact same join logic from your original sceene_description_with_tts)
        text_queue.join()
        audio_queue.join()
        wait_timeout=5
        if audio_generator_thread.is_alive(): audio_generator_thread.join(timeout=wait_timeout)
        if audio_player_thread.is_alive(): audio_player_thread.join(timeout=wait_timeout)
        print("Scene description audio finished.")
        print(f"--- Scene description complete for: {image_path} ---")

    return full_response_text # Return the complete text

# --- Main Execution ---
if __name__ == "__main__":
    try:
        # Set path for pytesseract if needed (often required on Windows)
        # pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe' # Example path
        assistant = Assistant()
        assistant.run()
    except RuntimeError as e:
        print(f"Initialization failed: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
# Assume NodeMapManager class exists as defined in the first snippet
# (with _load_map, _build_graph, get_shortest_path, get_node_data, calculate_distance)
# Let's re-add it here for completeness (slightly adapted)

import json
import networkx as nx
import math
import os
import threading
import queue
import time
import re
import sounddevice as sd
import numpy as np
from PIL import Image
import pytesseract

# Your existing utils (assuming they exist)
from utils.model import initiate_llm, initiate_tts_model
from utils.to_base64 import encode_image_to_base64
from utils.audio_player import audio_player
from utils.audio_generator import audio_generator
from utils.snap_a_picture import capture_image

# LangChain imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

# --- NodeMapManager Class ---
class NodeMapManager:
    def __init__(self, map_filepath):
        if not os.path.exists(map_filepath):
             raise FileNotFoundError(f"Map file not found: {map_filepath}")
        self.map_data = self._load_map(map_filepath)
        self.graph = nx.DiGraph() # Use DiGraph for potential one-way paths
        self._build_graph()
        self._node_name_to_id = {
            str(node.get('name', '')).lower(): str(node['id'])
            for node in self.map_data.get('nodes', []) if 'name' in node and 'id' in node
        }
        print(f"Loaded map with {self.graph.number_of_nodes()} nodes and {self.graph.number_of_edges()} edges.")
        print(f"Node name map: {self._node_name_to_id}")


    def _load_map(self, filepath):
        with open(filepath, 'r') as f:
            map_data = json.load(f)
        return map_data

    def get_shortest_path(self, start_node_id, end_node_id):
        try:
            # Use Dijkstra's algorithm (shortest_path defaults to it for weighted graphs)
            shortest_path = nx.shortest_path(self.graph, source=start_node_id, target=end_node_id, weight='weight')
            return shortest_path
        except nx.NetworkXNoPath:
            print(f"No path found between {start_node_id} and {end_node_id}")
            return None
        except nx.NodeNotFound as e:
            print(f"Node not found in graph: {e}")
            return None

    def get_node_data(self, node_id):
        try:
            return self.graph.nodes[str(node_id)] # Ensure node_id is string
        except KeyError:
            print(f"Node ID {node_id} not found in graph nodes.")
            return None

    def get_node_id_by_name(self, name: str) -> str | None:
         """Finds node ID by its 'name' attribute (case-insensitive)."""
         return self._node_name_to_id.get(name.lower())


    def _build_graph(self):
        if 'nodes' not in self.map_data:
            print("Warning: No 'nodes' found in map data.")
            return

        # Add nodes
        for node_data in self.map_data['nodes']:
            node_id = str(node_data['id'])
            # Store all attributes from JSON, ensure x, y exist
            node_attrs = node_data.copy()
            node_attrs['x'] = node_attrs.get('x', 0) # Default coordinates if missing
            node_attrs['y'] = node_attrs.get('y', 0)
            self.graph.add_node(node_id, **node_attrs)

        # Add edges (using outgoingLinks if available, otherwise assume bidirectional)
        if 'links' in self.map_data: # Check for explicit links/edges array first
            for link in self.map_data['links']:
                start_node = str(link.get('startNode'))
                end_node = str(link.get('endNode'))
                if self.graph.has_node(start_node) and self.graph.has_node(end_node):
                    distance = self.calculate_distance_by_ids(start_node, end_node)
                    self.graph.add_edge(start_node, end_node, weight=distance)
                    # Add reverse edge if links are meant to be bidirectional and not specified twice
                    if not self.graph.has_edge(end_node, start_node):
                         self.graph.add_edge(end_node, start_node, weight=distance)

        # Fallback or addition: use outgoingLinks within nodes
        for node_data in self.map_data['nodes']:
             node_id = str(node_data['id'])
             if 'outgoingLinks' in node_data:
                 for link in node_data['outgoingLinks']:
                     end_node = str(link.get('endNode'))
                     if self.graph.has_node(node_id) and self.graph.has_node(end_node):
                         if not self.graph.has_edge(node_id, end_node): # Avoid overwriting if already added
                            distance = self.calculate_distance_by_ids(node_id, end_node)
                            self.graph.add_edge(node_id, end_node, weight=distance)


    def calculate_distance(self, node1_data, node2_data):
        """Calculates Euclidean distance between two nodes given their data dictionaries."""
        x1 = node1_data.get('x', 0)
        y1 = node1_data.get('y', 0)
        x2 = node2_data.get('x', 0)
        y2 = node2_data.get('y', 0)
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    def calculate_distance_by_ids(self, node1_id, node2_id):
         """Calculates distance between two nodes given their IDs."""
         node1_data = self.get_node_data(node1_id)
         node2_data = self.get_node_data(node2_id)
         if node1_data and node2_data:
             return self.calculate_distance(node1_data, node2_data)
         else:
             print(f"Warning: Could not calculate distance between {node1_id} and {node2_id} due to missing node data.")
             return float('inf') # Return infinity if nodes are missing


# --- Placeholder for STT ---
def listen_for_command():
    print("Listening for voice command...")
    time.sleep(1) # Simulate listening
    # command = input("Enter command (or 'quit'): ") # Manual input
    # command = "describe the scene"
    # command = "read the sign"
    command = "how do I get to the elevator"
    # command = "navigate to room 101"
    # command = "quit"
    print(f"Heard: '{command}'")
    return command

# --- Placeholder for Positioning ---
def get_current_location_node_id(node_map_manager: NodeMapManager) -> str | None:
    """Placeholder: Estimate user position and return the NEAREST node ID."""
    # Simulate user being near the 'Entrance' node
    user_x, user_y = 5, 5 # Simulated coordinates
    min_dist = float('inf')
    nearest_node_id = None
    for node_id, node_data in node_map_manager.graph.nodes(data=True):
        dist = node_map_manager.calculate_distance({'x': user_x, 'y': user_y}, node_data)
        if dist < min_dist:
            min_dist = dist
            nearest_node_id = node_id
    print(f"DEBUG: Simulated user location maps to nearest node: {nearest_node_id} (Dist: {min_dist:.2f})")
    return nearest_node_id

# --- Placeholder for Orientation ---
def get_current_heading() -> float:
    """Placeholder: Get user's current heading in degrees (0=East, 90=North, 180=West, 270=South)."""
    # Simulate user facing mostly North
    heading = 85.0
    print(f"DEBUG: Simulated user heading: {heading:.1f} degrees")
    return heading


# --- Navigation Instruction Generation ---
def calculate_angle_and_turn(heading1, heading2):
    """Calculates the relative turn angle from heading1 to heading2."""
    # Normalize angles to 0-360
    h1 = heading1 % 360
    h2 = heading2 % 360
    delta = h2 - h1
    if delta > 180:
        delta -= 360
    elif delta <= -180:
        delta += 360
    return delta # Angle in degrees (-180 to 180)

def get_relative_turn_command(turn_angle_deg):
    """Converts a turn angle into a human-readable command."""
    abs_angle = abs(turn_angle_deg)
    if abs_angle < 15:
        return "Continue straight"
    elif abs_angle < 60:
        direction = "right" if turn_angle_deg > 0 else "left"
        return f"Turn slightly {direction}"
    elif abs_angle < 120:
        direction = "right" if turn_angle_deg > 0 else "left"
        return f"Turn {direction}"
    elif abs_angle < 165:
        direction = "right" if turn_angle_deg > 0 else "left"
        return f"Turn sharply {direction}"
    else:
        return "Make a U-turn"

def get_vector_heading(x1, y1, x2, y2):
    """Calculates the heading angle of the vector from (x1, y1) to (x2, y2)."""
    angle_rad = math.atan2(y2 - y1, x2 - x1) # Note: atan2(y, x)
    angle_deg = math.degrees(angle_rad)
    return angle_deg % 360 # Normalize to 0-360

def generate_navigation_instructions(path: list[str], node_map_manager: NodeMapManager, initial_heading: float) -> list[str]:
    """Generates turn-by-turn instructions from a path of node IDs."""
    if not path or len(path) < 2:
        return ["No path or path too short for instructions."]

    instructions = []
    current_heading = initial_heading

    for i in range(len(path) - 1):
        current_node_id = path[i]
        next_node_id = path[i+1]
        node_after_next_id = path[i+2] if i + 2 < len(path) else None

        current_node_data = node_map_manager.get_node_data(current_node_id)
        next_node_data = node_map_manager.get_node_data(next_node_id)

        if not current_node_data or not next_node_data:
            instructions.append(f"Error: Missing data for nodes {current_node_id} or {next_node_id}.")
            continue

        # Calculate segment details
        target_heading = get_vector_heading(current_node_data['x'], current_node_data['y'], next_node_data['x'], next_node_data['y'])
        distance = node_map_manager.calculate_distance(current_node_data, next_node_data)
        turn_angle = calculate_angle_and_turn(current_heading, target_heading)
        turn_command = get_relative_turn_command(turn_angle)

        # Build instruction
        instruction = ""
        if i == 0: # First step
            instruction += f"Start by heading towards {next_node_data.get('name', 'node ' + next_node_id)}. "
            if turn_command != "Continue straight":
                 instruction += f"{turn_command}. "
        else: # Subsequent steps
             instruction += f"{turn_command} towards {next_node_data.get('name', 'node ' + next_node_id)}. "

        instruction += f"Proceed for approximately {distance:.1f} meters."

        # Add landmark info if available
        if 'landmark' in next_node_data:
             instruction += f" You should pass {next_node_data['landmark']}."
        elif 'type' in next_node_data and next_node_data['type'] != 'Intersection': # Don't mention boring types
             instruction += f" It's near the {next_node_data['type']}."

        # Add anticipation for the *next* turn
        if node_after_next_id:
            node_after_next_data = node_map_manager.get_node_data(node_after_next_id)
            if node_after_next_data:
                heading_after_next = get_vector_heading(next_node_data['x'], next_node_data['y'], node_after_next_data['x'], node_after_next_data['y'])
                next_turn_angle = calculate_angle_and_turn(target_heading, heading_after_next)
                next_turn_command = get_relative_turn_command(next_turn_angle)
                if next_turn_command != "Continue straight":
                     instruction += f" Then, prepare to {next_turn_command.lower()}."

        instructions.append(instruction)
        current_heading = target_heading # Update heading for the next calculation

    # Add arrival message
    destination_node_data = node_map_manager.get_node_data(path[-1])
    if destination_node_data:
        instructions.append(f"You should arrive at your destination: {destination_node_data.get('name', 'node ' + path[-1])}.")
    else:
         instructions.append("You should arrive at your destination.")

    return instructions


# --- Assistant Class (Modified) ---
class Assistant:
    def __init__(self, map_filepath="map.json"): # Add map path argument
        print("Initializing Assistant...")
        # --- Config ---
        self.lm_studio_base_url = "http://localhost:1234/v1"
        self.tool_llm_model_id = "hugging-quants/Llama-3.2-3B-Instruct-Q8_0-GGUF/llama-3.2-3b-instruct-q8_0.gguf"
        self.vision_llm_model_id = "lmstudio-community/granite-vision-3.2-2b-GGUF"
        os.environ["OPENAI_API_KEY"] = "lm-studio"
        self.tts_voice = 'am_onyx'
        self.audio_sample_rate = 24000

        # --- Load Map ---
        print(f"Loading map from {map_filepath}...")
        try:
            self.node_map_manager = NodeMapManager(map_filepath)
        except FileNotFoundError as e:
             print(f"Error: {e}. Navigation will be unavailable.")
             self.node_map_manager = None # Indicate map is unavailable
        except Exception as e:
             print(f"Error loading or building map graph: {e}. Navigation will be unavailable.")
             self.node_map_manager = None


        # --- Load Models ---
        print("Initializing TTS...")
        self.tts_pipeline = initiate_tts_model(desired_device='cuda')
        if not self.tts_pipeline:
            raise RuntimeError("Failed to initialize TTS pipeline. Exiting.")

        print("Initializing Tool-Using LLM...")
        self.agent_llm = ChatOpenAI(
            base_url=self.lm_studio_base_url,
            model=self.tool_llm_model_id,
            temperature=0.1,
        )

        # --- Define Tools (linking methods of this class) ---
        @tool
        def scene_description_tool() -> str:
            """(Docstring unchanged)"""
            return self._execute_scene_description()

        @tool
        def ocr_text_tool() -> str:
            """(Docstring unchanged)"""
            return self._execute_ocr()

        @tool
        def indoor_navigation_tool(destination: str) -> str:
            """
            Calculates and provides turn-by-turn indoor navigation directions from the user's current estimated location to a known destination name within the building (e.g., 'reception', 'elevator', 'Room 101').
            """
            # Check if map loaded successfully
            if not self.node_map_manager:
                return "Sorry, the floor map is unavailable, so I cannot provide indoor navigation."

            # Get current location and heading (using placeholders)
            current_node_id = get_current_location_node_id(self.node_map_manager)
            current_head = get_current_heading() # Placeholder

            if not current_node_id:
                 return "Sorry, I couldn't determine your current location on the map."

            # Find destination node ID from name
            destination_node_id = self.node_map_manager.get_node_id_by_name(destination)
            if not destination_node_id:
                # Ask LLM to clarify if name is ambiguous or not found? Or just fail?
                return f"Sorry, I couldn't find '{destination}' on the map. Please try a known location name."

            return self._execute_indoor_navigation(current_node_id, destination_node_id, current_head)

        self.tools = [scene_description_tool, ocr_text_tool, indoor_navigation_tool]
        self.llm_with_tools = self.agent_llm.bind_tools(self.tools)
        self.chat_history = [SystemMessage(content="You are a helpful assistant for blind users. Use the available tools proactively to answer questions about surroundings, read text, or provide navigation. Use indoor_navigation_tool for requests like 'go to the elevator' or 'find Room 101'. Be concise but informative.")]

        print("Assistant Initialized.")

    # speak() method remains the same as before...
    def speak(self, text: str):
        """Uses the shared TTS pipeline to speak text."""
        if not text or not self.tts_pipeline:
            return
        print(f"Speaking: {text}")
        temp_audio_queue = queue.Queue()
        temp_text_queue = queue.Queue()
        temp_gen_done = threading.Event()
        temp_text_ready = threading.Event()
        temp_audio_playing = threading.Event()
        temp_text_queue.put(text)
        temp_gen_done.set()
        gen_thread = threading.Thread(target=audio_generator, args=(temp_text_queue, temp_audio_queue, temp_gen_done, temp_text_ready, self.tts_pipeline, self.tts_voice, 1), daemon=True)
        play_thread = threading.Thread(target=audio_player, args=(temp_audio_queue, temp_gen_done, temp_text_ready, temp_audio_playing, self.audio_sample_rate, 0.05, 0.1, 0.1), daemon=True)
        gen_thread.start()
        play_thread.start()
        gen_thread.join()
        play_thread.join()
        print("Speaking finished.")


    # _execute_scene_description() remains the same...
    def _execute_scene_description(self) -> str:
        self.speak("Okay, looking around...")
        # ... (rest of the implementation calling sceene_description_with_tts_modified)
        try:
             full_description = sceene_description_with_tts_modified(self.tts_pipeline, self.vision_llm_model_id, self.lm_studio_base_url)
             return full_description
        except Exception as e:
             error_msg = f"Sorry, I encountered an error describing the scene: {e}"
             print(error_msg)
             self.speak(error_msg)
             return error_msg

    # _execute_ocr() remains the same...
    def _execute_ocr(self) -> str:
        self.speak("Okay, trying to read...")
        # ... (rest of the implementation using capture_image and pytesseract)
        try:
            image_path = capture_image()
            if not image_path or not os.path.exists(image_path):
                raise FileNotFoundError("Failed to capture or find image.")
            image = Image.open(image_path)
            text = pytesseract.image_to_string(image)
            result_text = text.strip() if text.strip() else "No readable text found."
            # Only speak if text was found? Maybe speak "No text found" too.
            self.speak(result_text)
            return result_text
        except Exception as e:
            error_msg = f"Sorry, I encountered an error reading the text: {e}"
            print(error_msg)
            self.speak(error_msg)
            return error_msg


    # --- MODIFIED Navigation Execution ---
    def _execute_indoor_navigation(self, start_node_id: str, end_node_id: str, current_heading: float) -> str:
        """Internal: Calculates path and generates instructions."""
        if start_node_id == end_node_id:
            msg = "It looks like you are already at your destination."
            self.speak(msg)
            return msg

        start_node_name = self.node_map_manager.get_node_data(start_node_id).get('name', start_node_id)
        end_node_name = self.node_map_manager.get_node_data(end_node_id).get('name', end_node_id)
        self.speak(f"Okay, calculating route from {start_node_name} to {end_node_name}.")

        # 1. Calculate Path
        path = self.node_map_manager.get_shortest_path(start_node_id, end_node_id)

        if not path:
            msg = f"Sorry, I couldn't find a path from {start_node_name} to {end_node_name}."
            self.speak(msg)
            return msg

        # 2. Generate Instructions
        instructions = generate_navigation_instructions(path, self.node_map_manager, current_heading)

        # 3. Speak Instructions (For now, speak all at once - needs state machine for real-time)
        if instructions:
            # Combine instructions into a single string for now
            full_guidance = " ".join(instructions)
            print("\n--- Generated Navigation Instructions ---")
            for i, step in enumerate(instructions):
                print(f"{i+1}. {step}")
            print("-----------------------------------------\n")

            # Speak the first instruction clearly, maybe summarize the rest?
            # For now, just speak the first step. A real system needs the state machine.
            self.speak(instructions[0])
            # self.speak(full_guidance) # Option: Speak everything (can be long)

            # Return confirmation to agent
            return f"Route calculated. Starting guidance to {end_node_name}. First step: {instructions[0]}"
        else:
            msg = "Found a path, but couldn't generate instructions."
            self.speak(msg)
            return msg

    # run() method remains largely the same, agent handles calling the right tool...
    def run(self):
        """Main interaction loop."""
        self.speak("Assistant ready. How can I help?")
        while True:
            try:
                command = listen_for_command()
                if not command:
                    continue
                if command.lower() == 'quit':
                    self.speak("Goodbye!")
                    break

                self.chat_history.append(HumanMessage(content=command))

                print("Invoking agent...")
                ai_response = self.llm_with_tools.invoke(self.chat_history)
                self.chat_history.append(ai_response)

                final_response_to_speak = ""

                if ai_response.tool_calls:
                    print(f"Agent wants to call tools: {ai_response.tool_calls}")
                    tool_outputs = []
                    executed_tool_names = [] # Keep track of what ran
                    for tool_call in ai_response.tool_calls:
                        tool_name = tool_call['name']
                        tool_args = tool_call['args']
                        tool_id = tool_call['id']
                        executed_tool_names.append(tool_name)

                        selected_tool = next((t for t in self.tools if t.name == tool_name), None)
                        if selected_tool:
                            try:
                                # Tool execution happens here (calls _execute_... methods)
                                # The speak() calls happen *inside* the _execute methods now
                                tool_result = selected_tool.invoke(tool_args)
                                tool_outputs.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))
                            except Exception as e:
                                print(f"Error executing tool {tool_name}: {e}")
                                tool_outputs.append(ToolMessage(content=f"Error executing tool {tool_name}: {e}", tool_call_id=tool_id))
                        else:
                             tool_outputs.append(ToolMessage(content=f"Error: Tool '{tool_name}' not found.", tool_call_id=tool_id))

                    # Get final summary from LLM based on tool output
                    if tool_outputs:
                        self.chat_history.extend(tool_outputs)
                        print("Sending tool results back to agent...")
                        final_ai_response = self.llm_with_tools.invoke(self.chat_history)
                        self.chat_history.append(final_ai_response)
                        final_response_to_speak = final_ai_response.content
                    else: # Should not happen if tool_calls existed, but safety check
                         final_response_to_speak = "There was an issue processing the tool request."

                else: # No tool call
                    final_response_to_speak = ai_response.content

                # Speak the final response from the LLM, *unless* the tool handled all speech.
                # Scene desc and OCR speak their main content directly.
                # Navigation speaks the first step.
                # So, the LLM's final summary is usually good to speak.
                if final_response_to_speak:
                     self.speak(final_response_to_speak)

            except Exception as e:
                print(f"An error occurred in the main loop: {e}")
                self.speak("Sorry, something went wrong.")
            finally:
                 if len(self.chat_history) > 10:
                    self.chat_history = self.chat_history[:1] + self.chat_history[-9:]


# --- Need the modified sceene_description_with_tts_modified ---
# (Assume it's defined as in the previous response)
def sceene_description_with_tts_modified(tts_pipeline, vision_model_id, base_url):
     # ... (Implementation from previous response) ...
    image_path = capture_image()
    print(f"--- Starting Scene Description with TTS for: {image_path} ---")
    if tts_pipeline is None: return "Error: TTS pipeline missing."
    if not image_path or not os.path.exists(image_path): return "Error: Failed to get image."
    audio_queue = queue.Queue(); text_queue = queue.Queue(); text_ready = threading.Event()
    generation_done = threading.Event(); audio_playing = threading.Event(); full_response_text = ""
    print(f"Initializing Vision LLM ({vision_model_id})...")
    try:
        llm = initiate_llm(base_url, vision_model_id, temperature=0.5); print("Vision LLM initialized.")
    except Exception as e: return f"Error initializing vision model: {e}"
    base64_image_data_uri = encode_image_to_base64(image_path)
    if not base64_image_data_uri: return "Error encoding image."
    prompt = "Describe this image in detail for a person who is blind. Focus on object positions, types, and potential obstacles or points of interest. Be descriptive."
    message = HumanMessage(content=[{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": base64_image_data_uri}}])
    audio_player_thread = threading.Thread(target=audio_player, args=(audio_queue, generation_done, text_ready, audio_playing, 24000, 0.05, 0.1, 0.1), daemon=True)
    audio_generator_thread = threading.Thread(target=audio_generator, args=(text_queue, audio_queue, generation_done, text_ready, tts_pipeline, 'am_onyx', 3), daemon=True)
    audio_generator_thread.start(); audio_player_thread.start()
    print("\nProcessing Vision LLM stream...")
    sentence_pattern = re.compile(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<![A-Z]\.)(?<=[.?!])\s+|\n\s*'); accumulated_text = ""
    try:
        for chunk in llm.stream([message]):
            if chunk.content:
                print(chunk.content, end="", flush=True); content = chunk.content
                accumulated_text += content; full_response_text += content
                parts = sentence_pattern.split(accumulated_text)
                if len(parts) > 1:
                     for i in range(len(parts) - 1):
                         # Simplified chunking - needs punctuation logic restored from original
                         sentence_chunk = parts[i].strip()
                         if sentence_chunk and len(sentence_chunk.split()) >= 3: text_queue.put(sentence_chunk)
                     accumulated_text = parts[-1]
        final_chunk = accumulated_text.strip()
        if final_chunk and len(final_chunk.split()) >= 3: text_queue.put(final_chunk)
    except Exception as e: print(f"\nError during Vision LLM streaming: {e}")
    finally:
        print("\nVision LLM stream finished."); generation_done.set()
        print("Waiting for scene description audio..."); text_queue.join(); audio_queue.join()
        wait_timeout=5
        if audio_generator_thread.is_alive(): audio_generator_thread.join(timeout=wait_timeout)
        if audio_player_thread.is_alive(): audio_player_thread.join(timeout=wait_timeout)
        print("Scene description audio finished.")
        print(f"--- Scene description complete for: {image_path} ---")
    return full_response_text

# --- Sample map.json file content ---
# Create a file named map.json in the same directory with this content:
"""
{
  "nodes": [
    {
      "id": "node1",
      "name": "Entrance",
      "type": "Door",
      "x": 0,
      "y": 0
    },
    {
      "id": "node2",
      "name": "Main Hallway Intersection",
      "type": "Intersection",
      "x": 20,
      "y": 0,
       "landmark": "a large potted plant"
    },
    {
      "id": "node3",
      "name": "Reception Desk Area",
      "type": "Area",
      "x": 40,
      "y": 0
    },
    {
      "id": "node4",
      "name": "Elevator Lobby",
      "type": "Elevator",
      "x": 40,
      "y": -10
    },
    {
      "id": "node5",
      "name": "Room 101 Door",
      "type": "Door",
      "x": 20,
      "y": 15
    },
     {
      "id": "node6",
      "name": "Room 101 Center",
      "type": "Room",
      "x": 25,
      "y": 15
    }
  ],
  "links": [
    {"startNode": "node1", "endNode": "node2"},
    {"startNode": "node2", "endNode": "node3"},
    {"startNode": "node3", "endNode": "node4"},
    {"startNode": "node2", "endNode": "node5"},
    {"startNode": "node5", "endNode": "node6"}
  ]
}
"""

# --- Main Execution ---
if __name__ == "__main__":
    try:
        # Set path for pytesseract if needed
        # pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
        assistant = Assistant(map_filepath="map.json") # Specify map file
        assistant.run()
    except RuntimeError as e:
        print(f"Initialization failed: {e}")
    except FileNotFoundError as e:
         print(f"Critical Error: {e}. Please ensure map.json exists.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


In [ ]:
# Assume NodeMapManager class exists as defined previously
# Assume audio/TTS utils exist
# Assume model/LLM utils exist
# Assume capture_image util exists

import json
import networkx as nx
import math
import os
import threading
import queue
import time
import re
import sounddevice as sd
import numpy as np
from PIL import Image
import pytesseract

# Your existing utils (assuming they exist)
from utils.model import initiate_llm, initiate_tts_model
from utils.to_base64 import encode_image_to_base64
from utils.audio_player import audio_player
from utils.audio_generator import audio_generator
from utils.snap_a_picture import capture_image

# LangChain imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

# --- NodeMapManager Class (Keep as defined in the previous version) ---
class NodeMapManager:
    def __init__(self, map_filepath):
        if not os.path.exists(map_filepath):
             raise FileNotFoundError(f"Map file not found: {map_filepath}")
        self.map_data = self._load_map(map_filepath)
        self.graph = nx.DiGraph() # Use DiGraph for potential one-way paths
        self._build_graph()
        self._node_name_to_id = {
            str(node.get('name', '')).lower(): str(node['id'])
            for node in self.map_data.get('nodes', []) if 'name' in node and 'id' in node
        }
        print(f"Loaded map with {self.graph.number_of_nodes()} nodes and {self.graph.number_of_edges()} edges.")
        # print(f"Node name map: {self._node_name_to_id}") # Less verbose


    def _load_map(self, filepath):
        with open(filepath, 'r') as f:
            map_data = json.load(f)
        return map_data

    def get_shortest_path(self, start_node_id, end_node_id):
        try:
            shortest_path = nx.shortest_path(self.graph, source=str(start_node_id), target=str(end_node_id), weight='weight')
            return shortest_path
        except nx.NetworkXNoPath:
            print(f"No path found between {start_node_id} and {end_node_id}")
            return None
        except nx.NodeNotFound as e:
            print(f"Node not found in graph: {e}")
            return None

    def get_node_data(self, node_id):
        try:
            return self.graph.nodes[str(node_id)] # Ensure node_id is string
        except KeyError:
            # print(f"Node ID {node_id} not found in graph nodes.") # Less verbose
            return None

    def get_node_id_by_name(self, name: str) -> str | None:
         return self._node_name_to_id.get(name.lower())

    def _build_graph(self):
        if 'nodes' not in self.map_data: return
        for node_data in self.map_data['nodes']:
            node_id = str(node_data['id']); node_attrs = node_data.copy()
            node_attrs['x'] = node_attrs.get('x', 0); node_attrs['y'] = node_attrs.get('y', 0)
            self.graph.add_node(node_id, **node_attrs)
        if 'links' in self.map_data:
            for link in self.map_data['links']:
                start_node = str(link.get('startNode')); end_node = str(link.get('endNode'))
                if self.graph.has_node(start_node) and self.graph.has_node(end_node):
                    distance = self.calculate_distance_by_ids(start_node, end_node)
                    if distance != float('inf'):
                        self.graph.add_edge(start_node, end_node, weight=distance)
                        if not self.graph.has_edge(end_node, start_node):
                             self.graph.add_edge(end_node, start_node, weight=distance)
        for node_data in self.map_data['nodes']:
             node_id = str(node_data['id'])
             if 'outgoingLinks' in node_data:
                 for link in node_data['outgoingLinks']:
                     end_node = str(link.get('endNode'))
                     if self.graph.has_node(node_id) and self.graph.has_node(end_node):
                         if not self.graph.has_edge(node_id, end_node):
                            distance = self.calculate_distance_by_ids(node_id, end_node)
                            if distance != float('inf'):
                                self.graph.add_edge(node_id, end_node, weight=distance)

    def calculate_distance(self, node1_data, node2_data):
        x1 = node1_data.get('x', 0); y1 = node1_data.get('y', 0)
        x2 = node2_data.get('x', 0); y2 = node2_data.get('y', 0)
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    def calculate_distance_by_ids(self, node1_id, node2_id):
         node1_data = self.get_node_data(node1_id); node2_data = self.get_node_data(node2_id)
         if node1_data and node2_data: return self.calculate_distance(node1_data, node2_data)
         else: return float('inf')


# --- Global Simulation State (for testing step-by-step) ---
# In a real system, this comes from the positioning module
SIMULATED_USER_NODE_ID = "node1"
SIMULATED_USER_HEADING = 85.0
SIMULATED_NAVIGATION_ACTIVE = False # Flag to control simulation movement
SIMULATED_TARGET_NODE_ID = None

def update_simulation_state(nav_active: bool, target_node_id: str | None):
    """Updates global simulation flags."""
    global SIMULATED_NAVIGATION_ACTIVE, SIMULATED_TARGET_NODE_ID
    SIMULATED_NAVIGATION_ACTIVE = nav_active
    SIMULATED_TARGET_NODE_ID = target_node_id
    if nav_active:
        print(f"SIM: Navigation active, target: {target_node_id}")
    else:
         print("SIM: Navigation inactive.")

def simulate_movement():
    """Simulates the user moving to the target node if navigation is active."""
    global SIMULATED_USER_NODE_ID, SIMULATED_USER_HEADING, SIMULATED_TARGET_NODE_ID
    if SIMULATED_NAVIGATION_ACTIVE and SIMULATED_TARGET_NODE_ID:
        # Simple jump: Assume user reaches the target after a short delay
        print(f"SIM: User moving from {SIMULATED_USER_NODE_ID} towards {SIMULATED_TARGET_NODE_ID}...")
        time.sleep(3) # Simulate time taken to move
        # Update heading based on the move (crude simulation)
        # In reality, this comes from IMU/VIO
        # old_node_data = assistant.node_map_manager.get_node_data(SIMULATED_USER_NODE_ID)
        # new_node_data = assistant.node_map_manager.get_node_data(SIMULATED_TARGET_NODE_ID)
        # if old_node_data and new_node_data:
        #      SIMULATED_USER_HEADING = get_vector_heading(old_node_data['x'], old_node_data['y'], new_node_data['x'], new_node_data['y'])

        SIMULATED_USER_NODE_ID = SIMULATED_TARGET_NODE_ID # Arrived!
        SIMULATED_TARGET_NODE_ID = None # Reset target until next instruction issued
        print(f"SIM: User arrived at {SIMULATED_USER_NODE_ID}. New Heading (Simulated): {SIMULATED_USER_HEADING:.1f}")


# --- Updated Placeholders using Simulation State ---
def get_current_location_node_id(node_map_manager: NodeMapManager) -> str | None:
    """Placeholder: Returns the globally simulated user node ID."""
    # In a real system, this would involve complex calculations based on sensor data
    # print(f"DEBUG: Reporting current location: {SIMULATED_USER_NODE_ID}") # Less verbose
    return SIMULATED_USER_NODE_ID

def get_current_heading() -> float:
    """Placeholder: Returns the globally simulated user heading."""
    # print(f"DEBUG: Reporting current heading: {SIMULATED_USER_HEADING:.1f}") # Less verbose
    return SIMULATED_USER_HEADING

# --- STT Placeholder ---
_command_queue = queue.Queue()

def listen_for_command_non_blocking():
    """Checks if a command was manually entered."""
    if not _command_queue.empty():
        return _command_queue.get()
    return None

def command_input_thread():
    """Thread to allow manual command input without blocking main loop."""
    time.sleep(5) # Give app time to start
    print("\n----- Enter commands ('nav elevator', 'describe', 'read', 'cancel', 'quit') -----")
    while True:
        try:
            command = input("> ")
            _command_queue.put(command)
            if command.lower() == 'quit':
                break
        except EOFError: # Handle pipe closing etc.
            _command_queue.put('quit')
            break

# --- Navigation Instruction Generation (Keep as defined previously) ---
def calculate_angle_and_turn(heading1, heading2):
    h1 = heading1 % 360; h2 = heading2 % 360; delta = h2 - h1
    if delta > 180: delta -= 360
    elif delta <= -180: delta += 360
    return delta

def get_relative_turn_command(turn_angle_deg):
    abs_angle = abs(turn_angle_deg)
    if abs_angle < 15: return "Continue straight"
    direction = "right" if turn_angle_deg > 0 else "left"
    if abs_angle < 60: return f"Turn slightly {direction}"
    if abs_angle < 120: return f"Turn {direction}"
    if abs_angle < 165: return f"Turn sharply {direction}"
    return "Make a U-turn"

def get_vector_heading(x1, y1, x2, y2):
    angle_rad = math.atan2(y2 - y1, x2 - x1); angle_deg = math.degrees(angle_rad)
    return angle_deg % 360

def generate_navigation_instructions(path: list[str], node_map_manager: NodeMapManager, initial_heading: float) -> list[str]:
    if not path or len(path) < 2: return []
    instructions = []; current_heading = initial_heading
    for i in range(len(path) - 1):
        current_node_id = path[i]; next_node_id = path[i+1]
        node_after_next_id = path[i+2] if i + 2 < len(path) else None
        current_node_data = node_map_manager.get_node_data(current_node_id)
        next_node_data = node_map_manager.get_node_data(next_node_id)
        if not current_node_data or not next_node_data: continue
        target_heading = get_vector_heading(current_node_data['x'], current_node_data['y'], next_node_data['x'], next_node_data['y'])
        distance = node_map_manager.calculate_distance(current_node_data, next_node_data)
        turn_angle = calculate_angle_and_turn(current_heading, target_heading)
        turn_command = get_relative_turn_command(turn_angle)
        instruction = ""
        next_node_name = next_node_data.get('name', 'node ' + next_node_id)
        if i == 0: instruction += f"Start by heading towards {next_node_name}. "
        else: instruction += f"{turn_command} towards {next_node_name}. "
        if i == 0 and turn_command != "Continue straight": instruction += f"{turn_command}. " # Add turn for first step if needed
        instruction += f"Proceed for approximately {distance:.1f} meters."
        if 'landmark' in next_node_data: instruction += f" You should pass {next_node_data['landmark']}."
        elif 'type' in next_node_data and next_node_data['type'] not in ['Intersection', 'Area']: instruction += f" It's near the {next_node_data['type']}."
        if node_after_next_id:
            node_after_next_data = node_map_manager.get_node_data(node_after_next_id)
            if node_after_next_data:
                heading_after_next = get_vector_heading(next_node_data['x'], next_node_data['y'], node_after_next_data['x'], node_after_next_data['y'])
                next_turn_angle = calculate_angle_and_turn(target_heading, heading_after_next)
                next_turn_command = get_relative_turn_command(next_turn_angle)
                if next_turn_command != "Continue straight": instruction += f" Then, prepare to {next_turn_command.lower()}."
        instructions.append(instruction); current_heading = target_heading
    destination_node_data = node_map_manager.get_node_data(path[-1])
    if destination_node_data: instructions.append(f"You should arrive at your destination: {destination_node_data.get('name', 'node ' + path[-1])}.")
    else: instructions.append("You should arrive at your destination.")
    return instructions

# --- Modified sceene_description_with_tts_modified ---
# (Assume it's defined as before - takes tts_pipeline, vision_model_id, base_url)
def sceene_description_with_tts_modified(tts_pipeline, vision_model_id, base_url):
     # ... (Implementation from previous response - IMPORTANT: Ensure it's here) ...
    image_path = capture_image()
    # print(f"--- Starting Scene Description with TTS for: {image_path} ---") # Less verbose
    if tts_pipeline is None: return "Error: TTS pipeline missing."
    if not image_path or not os.path.exists(image_path): return "Error: Failed to get image."
    audio_queue = queue.Queue(); text_queue = queue.Queue(); text_ready = threading.Event()
    generation_done = threading.Event(); audio_playing = threading.Event(); full_response_text = ""
    # print(f"Initializing Vision LLM ({vision_model_id})...") # Less verbose
    try: llm = initiate_llm(base_url, vision_model_id, temperature=0.5)
    except Exception as e: return f"Error initializing vision model: {e}"
    base64_image_data_uri = encode_image_to_base64(image_path)
    if not base64_image_data_uri: return "Error encoding image."
    prompt = "Describe this image in detail for a person who is blind. Focus on object positions, types, and potential obstacles or points of interest. Be descriptive."
    message = HumanMessage(content=[{"type": "text", "text": prompt}, {"type": "image_url", "image_url": {"url": base64_image_data_uri}}])
    audio_player_thread = threading.Thread(target=audio_player, args=(audio_queue, generation_done, text_ready, audio_playing, 24000, 0.05, 0.1, 0.1), daemon=True)
    audio_generator_thread = threading.Thread(target=audio_generator, args=(text_queue, audio_queue, generation_done, text_ready, tts_pipeline, 'am_onyx', 3), daemon=True)
    audio_generator_thread.start(); audio_player_thread.start()
    # print("\nProcessing Vision LLM stream...") # Less verbose
    sentence_pattern = re.compile(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<![A-Z]\.)(?<=[.?!])\s+|\n\s*'); accumulated_text = ""
    try:
        for chunk in llm.stream([message]):
            if chunk.content:
                # print(chunk.content, end="", flush=True); # Optional console stream
                content = chunk.content; accumulated_text += content; full_response_text += content
                parts = sentence_pattern.split(accumulated_text)
                if len(parts) > 1:
                     for i in range(len(parts) - 1):
                         sentence_chunk = parts[i].strip() # Needs punctuation logic restored if important
                         if sentence_chunk and len(sentence_chunk.split()) >= 3: text_queue.put(sentence_chunk)
                     accumulated_text = parts[-1]
        final_chunk = accumulated_text.strip()
        if final_chunk and len(final_chunk.split()) >= 3: text_queue.put(final_chunk)
    except Exception as e: print(f"\nError during Vision LLM streaming: {e}")
    finally:
        # print("\nVision LLM stream finished."); # Less verbose
        generation_done.set()
        # print("Waiting for scene description audio..."); # Less verbose
        text_queue.join(); audio_queue.join()
        wait_timeout=5
        if audio_generator_thread.is_alive(): audio_generator_thread.join(timeout=wait_timeout)
        if audio_player_thread.is_alive(): audio_player_thread.join(timeout=wait_timeout)
        # print("Scene description audio finished.") # Less verbose
        # print(f"--- Scene description complete for: {image_path} ---") # Less verbose
    return full_response_text

# --- Assistant Class (with Navigation State) ---
class Assistant:
    def __init__(self, map_filepath="map.json"):
        print("Initializing Assistant...")
        # --- Config ---
        self.lm_studio_base_url = "http://localhost:1234/v1"
        self.tool_llm_model_id = "hugging-quants/Llama-3.2-3B-Instruct-Q8_0-GGUF/llama-3.2-3b-instruct-q8_0.gguf"
        self.vision_llm_model_id = "lmstudio-community/granite-vision-3.2-2b-GGUF"
        os.environ["OPENAI_API_KEY"] = "lm-studio"
        self.tts_voice = 'am_onyx'
        self.audio_sample_rate = 24000
        self.navigation_update_interval = 2.0 # Seconds between navigation checks

        # --- Navigation State ---
        self.is_navigating = False
        self.navigation_path = []
        self.navigation_instructions = []
        self.current_nav_step = 0
        self.navigation_destination_name = ""
        self.last_nav_update_time = 0

        # --- Load Map ---
        print(f"Loading map from {map_filepath}...")
        try: self.node_map_manager = NodeMapManager(map_filepath)
        except Exception as e: print(f"Error loading map: {e}. Navigation unavailable."); self.node_map_manager = None

        # --- Load Models ---
        print("Initializing TTS..."); self.tts_pipeline = initiate_tts_model(desired_device='cuda')
        if not self.tts_pipeline: raise RuntimeError("TTS failed.")
        print("Initializing Tool-Using LLM..."); self.agent_llm = ChatOpenAI(base_url=self.lm_studio_base_url, model=self.tool_llm_model_id, temperature=0.1)

        # --- Define Tools ---
        @tool
        def scene_description_tool() -> str:
            """Describes the current scene using the camera."""
            if self.is_navigating: self.speak("Pausing navigation briefly to describe the scene.") # Optional feedback
            return self._execute_scene_description()
        @tool
        def ocr_text_tool() -> str:
            """Reads text in front of the user using the camera."""
            if self.is_navigating: self.speak("Pausing navigation briefly to read text.") # Optional feedback
            return self._execute_ocr()
        @tool
        def indoor_navigation_tool(destination: str) -> str:
            """Starts indoor navigation to a named location (e.g., 'reception', 'elevator')."""
            if self.is_navigating: return "You are already navigating. Please cancel the current route first to start a new one."
            if not self.node_map_manager: return "Map unavailable, cannot navigate."
            current_node_id = get_current_location_node_id(self.node_map_manager)
            current_head = get_current_heading()
            if not current_node_id: return "Cannot determine current location."
            destination_node_id = self.node_map_manager.get_node_id_by_name(destination)
            if not destination_node_id: return f"Cannot find '{destination}' on the map."
            return self._execute_indoor_navigation(current_node_id, destination_node_id, current_head)

        self.tools = [scene_description_tool, ocr_text_tool, indoor_navigation_tool]
        self.llm_with_tools = self.agent_llm.bind_tools(self.tools)
        self.chat_history = [SystemMessage(content="You are a helpful assistant for blind users providing scene descriptions, OCR, and step-by-step indoor navigation. Use tools proactively. Respond concisely.")]
        print("Assistant Initialized.")

    # --- Core Execution Methods ---
    def speak(self, text: str):
        """Uses the shared TTS pipeline to speak text."""
        # (Implementation remains the same - uses threads/queues)
        if not text or not self.tts_pipeline: return
        print(f"Speaking: {text}")
        temp_audio_queue = queue.Queue(); temp_text_queue = queue.Queue()
        temp_gen_done = threading.Event(); temp_text_ready = threading.Event(); temp_audio_playing = threading.Event()
        temp_text_queue.put(text); temp_gen_done.set()
        gen_thread = threading.Thread(target=audio_generator, args=(temp_text_queue, temp_audio_queue, temp_gen_done, temp_text_ready, self.tts_pipeline, self.tts_voice, 1), daemon=True)
        play_thread = threading.Thread(target=audio_player, args=(temp_audio_queue, temp_gen_done, temp_text_ready, temp_audio_playing, self.audio_sample_rate, 0.05, 0.1, 0.1), daemon=True)
        gen_thread.start(); play_thread.start(); gen_thread.join(); play_thread.join()
        # print("Speaking finished.") # Less verbose

    def _execute_scene_description(self) -> str:
        # self.speak("Okay, looking around...") # Speak call moved inside tool/LLM flow
        try: return sceene_description_with_tts_modified(self.tts_pipeline, self.vision_llm_model_id, self.lm_studio_base_url)
        except Exception as e: error_msg = f"Error describing scene: {e}"; print(error_msg); self.speak(error_msg); return error_msg

    def _execute_ocr(self) -> str:
        # self.speak("Okay, trying to read...") # Speak call moved inside tool/LLM flow
        try:
            image_path = capture_image()
            if not image_path or not os.path.exists(image_path): raise FileNotFoundError("Failed image capture.")
            text = pytesseract.image_to_string(Image.open(image_path)); result_text = text.strip() or "No readable text found."
            self.speak(result_text) # OCR speaks its direct result
            return result_text
        except Exception as e: error_msg = f"Error reading text: {e}"; print(error_msg); self.speak(error_msg); return error_msg

    # --- Navigation Logic ---
    def _execute_indoor_navigation(self, start_node_id: str, end_node_id: str, current_heading: float) -> str:
        """Calculates path, generates instructions, and STARTS the navigation session."""
        if start_node_id == end_node_id: return "Already at destination."

        start_node_name = self.node_map_manager.get_node_data(start_node_id).get('name', start_node_id)
        end_node_name = self.node_map_manager.get_node_data(end_node_id).get('name', end_node_id)
        print(f"Calculating route from {start_node_name} to {end_node_name}.")

        path = self.node_map_manager.get_shortest_path(start_node_id, end_node_id)
        if not path: return f"Cannot find path from {start_node_name} to {end_node_name}."

        instructions = generate_navigation_instructions(path, self.node_map_manager, current_heading)
        if not instructions: return "Path found, but failed to generate instructions."

        # Start the session
        self._start_navigation_session(path, instructions, end_node_name)

        # Return confirmation to the agent, including the first step
        return f"Starting navigation to {end_node_name}. First step: {self.navigation_instructions[0]}"

    def _start_navigation_session(self, path, instructions, destination_name):
        """Stores navigation data and speaks the first step."""
        self.navigation_path = path
        self.navigation_instructions = instructions
        self.current_nav_step = 0
        self.is_navigating = True
        self.navigation_destination_name = destination_name
        self.last_nav_update_time = time.time()

        print("\n--- Starting Navigation Session ---")
        print(f"Path: {' -> '.join(path)}")
        print("Instructions:")
        for i, step in enumerate(instructions): print(f"  {i}. {step}")
        print("-----------------------------------\n")

        # Update simulation state for testing
        next_target_node = self.navigation_path[self.current_nav_step + 1] if len(self.navigation_path) > 1 else None
        update_simulation_state(True, next_target_node)

        # Speak the first instruction
        self.speak(self.navigation_instructions[self.current_nav_step])

    def _cancel_navigation(self):
        """Stops the current navigation session."""
        if self.is_navigating:
            print("--- Cancelling Navigation ---")
            self.is_navigating = False
            self.navigation_path = []
            self.navigation_instructions = []
            self.current_nav_step = 0
            self.navigation_destination_name = ""
            update_simulation_state(False, None) # Update simulation state
            self.speak("Navigation cancelled.")
            return True
        return False

    def _advance_nav_step(self):
        """Moves to the next navigation step and speaks the instruction."""
        self.current_nav_step += 1
        if self.current_nav_step < len(self.navigation_instructions):
            instruction = self.navigation_instructions[self.current_nav_step]
            is_arrival_message = (self.current_nav_step == len(self.navigation_instructions) - 1)

            print(f"--- Advancing Navigation to Step {self.current_nav_step} ---")
            self.speak(instruction)

            if is_arrival_message:
                print("--- Navigation Complete ---")
                self.speak("Navigation complete.")
                # Reset state after speaking final instruction
                self.is_navigating = False
                update_simulation_state(False, None)
            else:
                 # Update simulation target for the *next* step
                next_target_node = self.navigation_path[self.current_nav_step + 1] if self.current_nav_step + 1 < len(self.navigation_path) else None
                update_simulation_state(True, next_target_node)
        else:
            # Should have been caught by the arrival message check, but as safety:
            print("--- Navigation Already Complete ---")
            self.is_navigating = False
            update_simulation_state(False, None)


    def _update_navigation_progress(self):
        """Checks user location against the path and advances steps if needed."""
        if not self.is_navigating or not self.node_map_manager:
            return

        # --- Simple check based on simulated node arrival ---
        current_loc_id = get_current_location_node_id(self.node_map_manager)

        # Expected target node for the *current* step instruction
        if self.current_nav_step + 1 < len(self.navigation_path):
            target_node_id = self.navigation_path[self.current_nav_step + 1]
        else: # Already at the last step (arrival message)
             # print("DEBUG Nav: Already on arrival step.")
             return

        # Check if user has 'arrived' at the target node for the current step
        if current_loc_id == target_node_id:
            print(f"DEBUG Nav: User detected at target node {target_node_id} for step {self.current_nav_step}.")
            self._advance_nav_step()
            self.last_nav_update_time = time.time() # Reset timer after advancing
        else:
            # User hasn't reached the target node yet.
            # Optional: Add distance checks or reminders here if needed.
            # Optional: Check for off-route status more robustly.
            # current_expected_node = self.navigation_path[self.current_nav_step]
            # if current_loc_id != current_expected_node:
            #      print(f"WARNING: User location {current_loc_id} doesn't match expected start node {current_expected_node} for step {self.current_nav_step}. Potentially off-route.")
            #      # Consider triggering re-routing here
            pass

    # --- Main Loop ---
    def run(self):
        """Main interaction loop with navigation state checks."""
        self.speak("Assistant ready.")
        input_thread = threading.Thread(target=command_input_thread, daemon=True)
        input_thread.start()

        last_command_check_time = time.time()
        command_check_interval = 0.5 # Check for input more frequently

        while True:
            current_time = time.time()

            # --- Navigation Update Logic ---
            if self.is_navigating:
                # Simulate user movement (in real system, positioning module runs independently)
                simulate_movement() # Moves SIMULATED_USER_NODE_ID if target is set

                # Check progress periodically
                if current_time - self.last_nav_update_time > self.navigation_update_interval:
                    # print("DEBUG: Checking navigation progress...") # Can be noisy
                    self._update_navigation_progress()
                    self.last_nav_update_time = current_time # Update time even if no step change

            # --- Command Processing Logic ---
            if current_time - last_command_check_time > command_check_interval:
                 command = listen_for_command_non_blocking() # Check queue
                 last_command_check_time = current_time

                 if command:
                    print(f"Processing Command: '{command}'")
                    if command.lower() == 'quit':
                        if self.is_navigating: self._cancel_navigation()
                        self.speak("Goodbye!")
                        break
                    elif command.lower() == 'cancel' or command.lower() == 'cancel navigation':
                        if not self._cancel_navigation():
                             self.speak("There is no active navigation to cancel.")
                        continue # Skip LLM call after cancelling

                    # --- Interact with LLM Agent ---
                    try:
                        self.chat_history.append(HumanMessage(content=command))
                        print("Invoking agent...")
                        ai_response = self.llm_with_tools.invoke(self.chat_history)
                        self.chat_history.append(ai_response)

                        final_response_to_speak = ""

                        if ai_response.tool_calls:
                            print(f"Agent wants to call tools: {ai_response.tool_calls}")
                            tool_outputs = []
                            executed_tool_names = []
                            for tool_call in ai_response.tool_calls:
                                tool_name = tool_call['name']; tool_args = tool_call['args']; tool_id = tool_call['id']
                                executed_tool_names.append(tool_name)
                                selected_tool = next((t for t in self.tools if t.name == tool_name), None)
                                if selected_tool:
                                    try:
                                        # Tool execution methods (_execute_...) handle their own primary speech
                                        tool_result = selected_tool.invoke(tool_args)
                                        tool_outputs.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))
                                    except Exception as e: print(f"Error executing tool {tool_name}: {e}"); tool_outputs.append(ToolMessage(content=f"Error: {e}", tool_call_id=tool_id))
                                else: tool_outputs.append(ToolMessage(content=f"Error: Tool '{tool_name}' not found.", tool_call_id=tool_id))

                            if tool_outputs:
                                self.chat_history.extend(tool_outputs)
                                print("Sending tool results back to agent...")
                                final_ai_response = self.llm_with_tools.invoke(self.chat_history)
                                self.chat_history.append(final_ai_response)
                                final_response_to_speak = final_ai_response.content
                        else: # No tool call
                            final_response_to_speak = ai_response.content

                        # Speak final LLM summary/response (if any)
                        if final_response_to_speak:
                            # Avoid redundant speech if a tool handled everything comprehensively
                            # (e.g., OCR speaks its result directly, SceneDesc streams speech)
                            # Navigation start message is returned by the tool but spoken by LLM here.
                            self.speak(final_response_to_speak)

                    except Exception as e:
                        print(f"An error occurred during agent interaction: {e}")
                        self.speak("Sorry, I encountered an issue.")
                    finally:
                         # Trim history
                         if len(self.chat_history) > 10: self.chat_history = self.chat_history[:1] + self.chat_history[-9:]

            # Prevent busy-waiting
            time.sleep(0.1)


# --- Sample map.json (Keep as before) ---
# ... (Ensure map.json file exists with the previous content) ...

# --- Main Execution ---
if __name__ == "__main__":
    assistant = None # Define assistant outside try block for potential cleanup
    try:
        assistant = Assistant(map_filepath="map.json")
        assistant.run()
    except RuntimeError as e:
        print(f"Initialization failed: {e}")
    except FileNotFoundError as e:
         print(f"Critical Error: {e}. Please ensure map.json exists.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        # Optional cleanup (e.g., close resources if needed)
        print("Exiting application.")
        # Ensure TTS engine is shutdown cleanly if necessary (pyttsx3 might need this)
        if assistant and hasattr(assistant, 'tts_pipeline') and assistant.tts_pipeline:
             # Add specific cleanup for your TTS library if needed
             pass

In [ ]:
# Standard Library Imports
import json
import math
import os
import threading
import queue
import time
import re
import logging
from typing import Optional, List, Dict, Any, Tuple

# Third-party Imports
import networkx as nx
import sounddevice as sd # Keep for PortAudioError check potentially
import numpy as np # Keep for TTS if needed by underlying lib
from PIL import Image
import pytesseract
import yaml # For config file

# LangChain Imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

# Your existing utils (assuming they exist and are correct)
# IMPORTANT: Ensure these utils are compatible with changes if any
from utils.model import initiate_llm, initiate_tts_model
from utils.to_base64 import encode_image_to_base64
from utils.audio_player import audio_player # Assumes this handles playback from queue
from utils.audio_generator import audio_generator # Assumes this generates audio to queue
from utils.snap_a_picture import capture_image

# --- Basic Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

# --- NodeMapManager Class (Keep as defined previously, minor logging) ---
class NodeMapManager:
    def __init__(self, map_filepath):
        logger.info(f"Initializing NodeMapManager with map: {map_filepath}")
        if not os.path.exists(map_filepath):
             logger.error(f"Map file not found: {map_filepath}")
             raise FileNotFoundError(f"Map file not found: {map_filepath}")
        self.map_data = self._load_map(map_filepath)
        self.graph = nx.DiGraph()
        self._build_graph()
        self._node_name_to_id = {
            str(node.get('name', '')).lower(): str(node['id'])
            for node in self.map_data.get('nodes', []) if 'name' in node and 'id' in node
        }
        logger.info(f"Loaded map with {self.graph.number_of_nodes()} nodes and {self.graph.number_of_edges()} edges.")

    def _load_map(self, filepath):
        try:
            with open(filepath, 'r') as f:
                map_data = json.load(f)
            return map_data
        except Exception as e:
            logger.exception(f"Failed to load or parse map file {filepath}")
            raise

    def get_shortest_path(self, start_node_id, end_node_id):
        try:
            path = nx.shortest_path(self.graph, source=str(start_node_id), target=str(end_node_id), weight='weight')
            logger.debug(f"Shortest path found: {' -> '.join(path)}")
            return path
        except nx.NetworkXNoPath:
            logger.warning(f"No path found between {start_node_id} and {end_node_id}")
            return None
        except nx.NodeNotFound as e:
            logger.warning(f"Node not found in graph during pathfinding: {e}")
            return None
        except Exception as e:
            logger.exception("Error during pathfinding")
            return None

    def get_node_data(self, node_id):
        try:
            return self.graph.nodes[str(node_id)]
        except KeyError:
            return None # Handled by caller, less verbose log

    def get_node_id_by_name(self, name: str) -> Optional[str]:
         return self._node_name_to_id.get(name.lower())

    def _build_graph(self):
        # (Keep graph building logic as before - ensure robustness)
        if 'nodes' not in self.map_data: return
        for node_data in self.map_data['nodes']:
            node_id = str(node_data['id']); node_attrs = node_data.copy()
            node_attrs['x'] = node_attrs.get('x', 0); node_attrs['y'] = node_attrs.get('y', 0)
            self.graph.add_node(node_id, **node_attrs)
        # Add edges - ensure distance calculation handles missing nodes
        edges_added = set()
        if 'links' in self.map_data:
            for link in self.map_data['links']:
                s = str(link.get('startNode')); e = str(link.get('endNode'))
                if self.graph.has_node(s) and self.graph.has_node(e):
                    dist = self.calculate_distance_by_ids(s, e)
                    if dist != float('inf'):
                        if (s, e) not in edges_added: self.graph.add_edge(s, e, weight=dist); edges_added.add((s, e))
                        if (e, s) not in edges_added: self.graph.add_edge(e, s, weight=dist); edges_added.add((e, s)) # Assume bidirectional
        # Add edges from outgoingLinks if not already present
        for node_data in self.map_data['nodes']:
            node_id = str(node_data['id'])
            if 'outgoingLinks' in node_data:
                for link in node_data['outgoingLinks']:
                    end_node = str(link.get('endNode'))
                    if self.graph.has_node(node_id) and self.graph.has_node(end_node):
                         if (node_id, end_node) not in edges_added:
                             dist = self.calculate_distance_by_ids(node_id, end_node)
                             if dist != float('inf'):
                                 self.graph.add_edge(node_id, end_node, weight=dist); edges_added.add((node_id, end_node))

    def calculate_distance(self, node1_data, node2_data):
        x1=node1_data.get('x',0); y1=node1_data.get('y',0); x2=node2_data.get('x',0); y2=node2_data.get('y',0)
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    def calculate_distance_by_ids(self, node1_id, node2_id):
         node1_data = self.get_node_data(node1_id); node2_data = self.get_node_data(node2_id)
         if node1_data and node2_data: return self.calculate_distance(node1_data, node2_data)
         else: return float('inf')

# --- Simulation State & Placeholders (Improved for Pause/Resume) ---
SIMULATED_USER_NODE_ID = "node1" # Start at entrance
SIMULATED_USER_HEADING = 90.0 # Start facing North
SIMULATED_MOVEMENT_PAUSED = False # Flag for pause testing
_sim_last_update_time = time.time()
_sim_target_node_id = None
_sim_movement_delay = 3.0 # Time to "reach" next node

def update_simulation_target(target_node_id: Optional[str]):
    global _sim_target_node_id, _sim_last_update_time
    if _sim_target_node_id != target_node_id:
        logger.debug(f"SIM: New target set to {target_node_id}")
        _sim_target_node_id = target_node_id
        _sim_last_update_time = time.time() # Reset timer when target changes

def simulate_movement(current_path: Optional[List[str]] = None):
    global SIMULATED_USER_NODE_ID, SIMULATED_USER_HEADING, _sim_last_update_time
    if SIMULATED_MOVEMENT_PAUSED or not _sim_target_node_id:
        return # Don't move if paused or no target

    current_time = time.time()
    if current_time - _sim_last_update_time >= _sim_movement_delay:
        logger.info(f"SIM: Moving from {SIMULATED_USER_NODE_ID} -> {_sim_target_node_id}")
        # Crude heading update (assume turns happen instantly at arrival)
        if current_path:
             try:
                  current_index = current_path.index(SIMULATED_USER_NODE_ID)
                  if current_index + 1 < len(current_path):
                        next_node_id = current_path[current_index+1]
                        # Update heading towards the node *after* the target we just reached
                        if current_index + 2 < len(current_path):
                            node_after_next_id = current_path[current_index+2]
                            # Need access to NodeMapManager here... difficult from global scope
                            # Simplification: Just point towards the target node for now
                            # SIMULATED_USER_HEADING = get_vector_heading(...)
                  SIMULATED_USER_NODE_ID = _sim_target_node_id # Arrived!
                  logger.info(f"SIM: User arrived at {SIMULATED_USER_NODE_ID}")
                  update_simulation_target(None) # Clear target until next instruction
             except (ValueError, IndexError):
                  logger.warning("SIM: Could not find current/next node in path for heading update.")
                  SIMULATED_USER_NODE_ID = _sim_target_node_id # Arrive anyway
                  update_simulation_target(None)
        else:
            SIMULATED_USER_NODE_ID = _sim_target_node_id # Arrive without path info
            update_simulation_target(None)


def get_current_location_node_id() -> Optional[str]:
    """Placeholder: Returns the globally simulated user node ID."""
    return SIMULATED_USER_NODE_ID # Assume positioning system gives nearest node ID

def get_current_heading() -> float:
    """Placeholder: Returns the globally simulated user heading."""
    return SIMULATED_USER_HEADING

# --- STT Placeholder (Keep as before) ---
_command_queue = queue.Queue()
def listen_for_command_non_blocking():
    if not _command_queue.empty(): return _command_queue.get()
    return None
def command_input_thread():
    time.sleep(5)
    print("\n----- Enter commands ('nav elevator', 'describe', 'read', 'pause', 'resume', 'where am i', 'cancel', 'quit') -----")
    while True:
        try: command = input("> "); _command_queue.put(command)
        except EOFError: _command_queue.put('quit'); break
        if command.lower() == 'quit': break

# --- Navigation Instruction Generation (Keep as defined previously) ---
# calculate_angle_and_turn, get_relative_turn_command, get_vector_heading, generate_navigation_instructions
# (Assume these functions exist exactly as in the previous step)
def calculate_angle_and_turn(h1, h2): # Condensed
    h1%=360; h2%=360; d=h2-h1; d-=(360 if d>180 else 0); d+=(360 if d<=-180 else 0); return d
def get_relative_turn_command(a): # Condensed
    b=abs(a); d="right" if a>0 else "left";
    if b<15: return "Continue straight";
    if b<60: return f"Turn slightly {d}";
    if b<120: return f"Turn {d}";
    if b<165: return f"Turn sharply {d}";
    return "Make a U-turn"
def get_vector_heading(x1,y1,x2,y2): return math.degrees(math.atan2(y2-y1,x2-x1))%360
def generate_navigation_instructions(path, node_map_manager, initial_heading): # Condensed structure
    if not path or len(path)<2: return []
    instr=[]; cur_h=initial_heading
    for i in range(len(path)-1):
        c_id=path[i]; n_id=path[i+1]; nn_id=path[i+2] if i+2<len(path) else None
        c_d=node_map_manager.get_node_data(c_id); n_d=node_map_manager.get_node_data(n_id)
        if not c_d or not n_d: instr.append(f"Err: Node {c_id}/{n_id}"); continue
        tgt_h=get_vector_heading(c_d['x'],c_d['y'],n_d['x'],n_d['y'])
        dist=node_map_manager.calculate_distance(c_d,n_d); turn_a=calculate_angle_and_turn(cur_h,tgt_h); turn_c=get_relative_turn_command(turn_a)
        s=""; n_name=n_d.get('name',f'node {n_id}');
        if i==0: s+=f"Start heading towards {n_name}. "; s+=f"{turn_c}. " if turn_c!="Continue straight" else ""
        else: s+=f"{turn_c} towards {n_name}. "
        s+=f"Proceed approx {dist:.1f}m."
        if 'landmark' in n_d: s+=f" Pass {n_d['landmark']}."
        elif n_d.get('type') not in [None,'Intersection','Area']: s+=f" Near the {n_d['type']}."
        if nn_id:
            nn_d=node_map_manager.get_node_data(nn_id)
            if nn_d:
                nn_h=get_vector_heading(n_d['x'],n_d['y'],nn_d['x'],nn_d['y']); nxt_t_a=calculate_angle_and_turn(tgt_h,nn_h)
                nxt_t_c=get_relative_turn_command(nxt_t_a);
                if nxt_t_c!="Continue straight": s+=f" Then prepare to {nxt_t_c.lower()}."
        instr.append(s); cur_h=tgt_h
    dest_d=node_map_manager.get_node_data(path[-1]); instr.append(f"Arrive at {dest_d.get('name',path[-1])}." if dest_d else "Arrive.")
    return instr


# --- sceene_description_with_tts_modified (Keep as defined previously) ---
# (Assume implementation exists)
def sceene_description_with_tts_modified(tts_pipeline, vision_model_id, base_url):
    # ... full implementation ...
    logger.info("Executing scene description")
    # ... rest of logic ...
    return "Scene description text placeholder" # Replace with actual return


# --- Navigation Manager ---
class NavigationManager:
    def __init__(self, assistant_speak_func, node_map_manager, config):
        self.speak = assistant_speak_func # Function to speak text
        self.node_map_manager = node_map_manager
        self.config = config
        self.is_navigating = False
        self.is_paused = False
        self.navigation_path: List[str] = []
        self.navigation_instructions: List[str] = []
        self.current_nav_step: int = 0
        self.navigation_destination_name: str = ""
        self.last_nav_update_time: float = 0
        self.off_route_counter: int = 0 # Count consecutive off-route checks

    def start_session(self, path, instructions, destination_name):
        if not path or not instructions:
            logger.error("Navigation session start failed: Missing path or instructions.")
            return False
        self.navigation_path = path
        self.navigation_instructions = instructions
        self.current_nav_step = 0
        self.is_navigating = True
        self.is_paused = False
        self.navigation_destination_name = destination_name
        self.last_nav_update_time = time.time()
        self.off_route_counter = 0

        logger.info(f"Starting navigation to {destination_name}. Path: {'->'.join(path)}")
        # Update simulation target for the first step
        next_target_node = self.navigation_path[1] if len(self.navigation_path) > 1 else None
        update_simulation_target(next_target_node)

        # Speak the first instruction
        self.speak(self.navigation_instructions[self.current_nav_step])
        return True

    def cancel_session(self, speak_cancel=True):
        if self.is_navigating:
            logger.info("Cancelling navigation.")
            self.is_navigating = False
            self.is_paused = False
            update_simulation_target(None) # Stop simulation movement
            if speak_cancel: self.speak("Navigation cancelled.")
            # Reset state fully
            self.navigation_path = []; self.navigation_instructions = []; self.current_nav_step = 0; self.navigation_destination_name = ""
            return True
        return False

    def pause_session(self):
        if self.is_navigating and not self.is_paused:
            logger.info("Pausing navigation.")
            self.is_paused = True
            global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = True # Pause simulation
            self.speak("Navigation paused.")
            return True
        return False

    def resume_session(self):
        if self.is_navigating and self.is_paused:
            logger.info("Resuming navigation.")
            self.is_paused = False
            global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = False # Resume simulation
            self.last_nav_update_time = time.time() # Reset timer
            self.speak(f"Resuming navigation. Current step: {self.navigation_instructions[self.current_nav_step]}")
             # Re-set simulation target
            next_target_node = self.navigation_path[self.current_nav_step + 1] if self.current_nav_step + 1 < len(self.navigation_path) else None
            update_simulation_target(next_target_node)
            return True
        return False

    def get_status(self):
        """Reports current navigation status."""
        if not self.is_navigating:
            return "Not currently navigating."
        if self.is_paused:
            return f"Navigation paused. Current step: {self.navigation_instructions[self.current_nav_step]}"

        status = f"Navigating towards {self.navigation_destination_name}. "
        status += f"Current instruction: {self.navigation_instructions[self.current_nav_step]}"
        # Add distance/ETA estimate here if using coordinates
        return status

    def repeat_instruction(self):
         if self.is_navigating:
              self.speak(f"Current instruction: {self.navigation_instructions[self.current_nav_step]}")
              return True
         return False

    def update_progress(self) -> Optional[str]:
        """Checks user location, advances steps, or triggers reroute. Returns 'reroute' if needed."""
        if not self.is_navigating or self.is_paused or not self.node_map_manager:
            return None

        current_time = time.time()
        if current_time - self.last_nav_update_time < self.config.get('navigation_update_interval', 2.0):
             return None # Don't check too frequently

        self.last_nav_update_time = current_time
        current_loc_id = get_current_location_node_id()

        if not current_loc_id:
            logger.warning("Nav Update: Could not get current location.")
            # Maybe speak "Lost tracking?" after several failures
            return None

        # Check arrival at current step's target node
        if self.current_nav_step + 1 < len(self.navigation_path):
            target_node_id = self.navigation_path[self.current_nav_step + 1]
            current_expected_node = self.navigation_path[self.current_nav_step]
        else: # Already on the final (arrival) instruction
            return None

        if current_loc_id == target_node_id:
            logger.info(f"Nav Update: User reached target {target_node_id} for step {self.current_nav_step}.")
            self._advance_step()
            self.off_route_counter = 0 # Reset counter on successful step
        elif current_loc_id != current_expected_node:
            # Basic Off-Route Check (Simulation based on unexpected node)
            self.off_route_counter += 1
            logger.warning(f"Nav Update: User at {current_loc_id}, expected {current_expected_node} or {target_node_id} (Off-route count: {self.off_route_counter})")
            if self.off_route_counter >= self.config.get('off_route_threshold_nodes', 2):
                 logger.warning("Triggering reroute due to being off-route.")
                 self.speak("It seems you are off the planned route. Let me recalculate.")
                 return "reroute" # Signal to Assistant to reroute
        else:
            # On the correct segment, but haven't reached target yet
            self.off_route_counter = 0 # Reset counter if back on track
            # logger.debug(f"Nav Update: User at {current_loc_id}, proceeding towards {target_node_id}.") # Too verbose

        return None # No action needed / handled internally

    def _advance_step(self):
        """Internal: Moves to next step, speaks instruction or handles arrival."""
        self.current_nav_step += 1
        if self.current_nav_step < len(self.navigation_instructions):
            instruction = self.navigation_instructions[self.current_nav_step]
            is_arrival = (self.current_nav_step == len(self.navigation_instructions) - 1)
            logger.info(f"Advancing to Nav Step {self.current_nav_step}: {instruction}")
            self.speak(instruction)

            if is_arrival:
                logger.info("Navigation complete.")
                self.speak("Navigation complete.")
                self.is_navigating = False; self.is_paused = False
                update_simulation_target(None)
            else:
                 # Update simulation target for the *next* step
                next_target_node = self.navigation_path[self.current_nav_step + 1] if self.current_nav_step + 1 < len(self.navigation_path) else None
                update_simulation_target(next_target_node)
        else:
            # Should be handled by arrival check, but safety fallback
            logger.warning("Tried to advance past the last instruction.")
            self.is_navigating = False; self.is_paused = False
            update_simulation_target(None)

# --- Speech Manager (for Async TTS) ---
class SpeechManager:
    def __init__(self, config):
        self.config = config
        self.tts_pipeline = None # Loaded later
        self.speak_queue = queue.Queue()
        self.speak_thread = threading.Thread(target=self._speech_worker, daemon=True)
        self.stop_event = threading.Event()
        self.current_playback_event = threading.Event() # To signal when playback finishes

    def load_tts(self):
        logger.info("Initializing TTS in SpeechManager...")
        self.tts_pipeline = initiate_tts_model(desired_device='cuda') # Or use config value
        if not self.tts_pipeline:
             logger.error("Failed to initialize TTS pipeline in SpeechManager.")
             raise RuntimeError("TTS failed.")
        self.speak_thread.start()
        logger.info("TTS loaded and speech worker started.")

    def speak(self, text: str, interrupt: bool = False):
        """Add text to the speech queue. If interrupt, clear queue first."""
        if not text: return
        if interrupt:
            logger.debug("Interrupting speech queue.")
            self._clear_queue()
        logger.debug(f"Queuing speech: '{text[:50]}...'")
        self.speak_queue.put(text)

    def _clear_queue(self):
        """Clears the speech queue."""
        while not self.speak_queue.empty():
            try: self.speak_queue.get_nowait()
            except queue.Empty: break
        # TODO: Need a way to stop the *currently playing* audio via audio_player if possible

    def _speech_worker(self):
        """Worker thread that processes the speech queue."""
        while not self.stop_event.is_set():
            try:
                text_to_speak = self.speak_queue.get(timeout=0.5) # Wait briefly for text
                logger.info(f"Speech Worker: Processing '{text_to_speak[:50]}...'")

                # Use audio_generator and audio_player utilities
                # This part assumes they work correctly with queues
                audio_q = queue.Queue()
                gen_done = threading.Event()
                text_ready = threading.Event() # Not strictly needed for single item
                audio_playing = threading.Event() # To track this specific playback

                # Need to handle the case where tts_pipeline isn't loaded yet
                if not self.tts_pipeline:
                    logger.error("Speech Worker: TTS pipeline not available!")
                    self.speak_queue.task_done()
                    continue

                # Create generator/player threads for this utterance
                gen_thread = threading.Thread(target=audio_generator, args=(queue.Queue(), audio_q, gen_done, text_ready, self.tts_pipeline, self.config.get('tts_voice','am_onyx'), 1), daemon=True)
                 # Put single item in temp text queue for generator
                _temp_text_q = queue.Queue(); _temp_text_q.put(text_to_speak); gen_thread._args = (_temp_text_q,) + gen_thread._args[1:] # Ugly hack to pass text

                play_thread = threading.Thread(target=audio_player, args=(audio_q, gen_done, text_ready, audio_playing, self.config.get('audio_sample_rate',24000), 0.05, 0.1, 0.1), daemon=True)

                gen_thread.start()
                play_thread.start()

                gen_done.set() # Signal generator no more text is coming for this utterance

                gen_thread.join()
                play_thread.join() # Wait for this specific utterance to finish playing
                logger.debug("Speech Worker: Finished utterance.")
                self.speak_queue.task_done()

            except queue.Empty:
                continue # No text in queue, loop again
            except Exception as e:
                logger.exception("Error in speech worker thread.")
                # Avoid crashing the thread if possible
                if not self.stop_event.is_set(): time.sleep(1)


    def wait_for_speech_to_finish(self):
        """Blocks until the speech queue is empty."""
        logger.debug("Waiting for speech queue to empty...")
        self.speak_queue.join()
        logger.debug("Speech queue empty.")
        # Add short delay to allow final playback buffer to clear?
        time.sleep(0.2)

    def shutdown(self):
        logger.info("Shutting down Speech Manager.")
        self.stop_event.set()
        # Add a final item to wake up the worker?
        self.speak_queue.put(None) # Sentinel value? Depends on worker loop
        if self.speak_thread.is_alive():
             self.speak_thread.join(timeout=2.0)
        logger.info("Speech Manager shutdown complete.")


# --- Assistant Class (Refactored) ---
class Assistant:
    def __init__(self, config_path="config.yaml"):
        logger.info("Initializing Assistant...")
        self.config = self._load_config(config_path)
        self._configure_libraries() # e.g., Tesseract path

        # --- Core Components ---
        self.node_map_manager = self._init_map_manager()
        self.speech_manager = SpeechManager(self.config) # Handles TTS loading & async speech
        self.navigation_manager = NavigationManager(self.speech_manager.speak, self.node_map_manager, self.config) if self.node_map_manager else None

        # --- LLM & Tools ---
        self.agent_llm = self._init_llm()
        self.tools = self._define_tools()
        self.llm_with_tools = self.agent_llm.bind_tools(self.tools)
        self.chat_history = [SystemMessage(content="You are a helpful assistant for blind users providing scene descriptions, OCR, and step-by-step indoor navigation. Use tools proactively. Respond concisely.")]

        logger.info("Assistant Initialized.")

    def _load_config(self, config_path):
        logger.info(f"Loading configuration from {config_path}")
        try:
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            # Set environment variables from config if needed
            os.environ["OPENAI_API_KEY"] = config.get('openai_api_key', 'lm-studio')
            return config
        except FileNotFoundError:
            logger.error(f"Configuration file not found: {config_path}")
            raise
        except Exception:
            logger.exception(f"Error loading configuration file: {config_path}")
            raise

    def _configure_libraries(self):
        # Example: Configure Tesseract path if specified in config
        tesseract_path = self.config.get('tesseract_cmd_path')
        if tesseract_path and os.path.exists(tesseract_path):
            pytesseract.pytesseract.tesseract_cmd = tesseract_path
            logger.info(f"Tesseract path set to: {tesseract_path}")
        elif tesseract_path:
             logger.warning(f"Tesseract path specified but not found: {tesseract_path}")

    def _init_map_manager(self):
        try:
            map_path = self.config.get('map_filepath', 'map.json')
            return NodeMapManager(map_path)
        except Exception as e:
            logger.error(f"Failed to initialize NodeMapManager: {e}. Navigation will be disabled.", exc_info=True)
            return None

    def _init_llm(self):
        logger.info("Initializing Tool-Using LLM...")
        try:
            return ChatOpenAI(
                base_url=self.config.get('lm_studio_base_url'),
                model=self.config.get('tool_llm_model_id'),
                temperature=0.1,
            )
        except Exception as e:
            logger.exception("Failed to initialize agent LLM.")
            raise RuntimeError("LLM init failed.") from e

    def _define_tools(self):
        # Define tools using methods of *this* Assistant class
        @tool
        def scene_description_tool() -> str:
            """Describes the current scene using the camera."""
            if self.navigation_manager and self.navigation_manager.is_navigating:
                # Optional feedback - maybe make speak non-blocking first
                # self.speech_manager.speak("Pausing navigation briefly...", interrupt=True)
                # self.navigation_manager.pause_session() # Decide if it should auto-pause
                pass
            return self._execute_scene_description()
        @tool
        def ocr_text_tool() -> str:
            """Reads text in front of the user using the camera."""
            # if self.navigation_manager and self.navigation_manager.is_navigating:
                # self.speech_manager.speak("Pausing navigation briefly...", interrupt=True)
                # self.navigation_manager.pause_session()
            return self._execute_ocr()
        @tool
        def indoor_navigation_tool(destination: str) -> str:
            """Starts indoor navigation to a named location (e.g., 'reception', 'elevator')."""
            return self._execute_indoor_navigation_request(destination)
        @tool
        def where_am_i_tool() -> str:
             """Reports the user's current estimated location or navigation status."""
             return self._execute_where_am_i()

        return [scene_description_tool, ocr_text_tool, indoor_navigation_tool, where_am_i_tool]

    # --- Tool Execution Logic ---
    def _execute_scene_description(self) -> str:
        # Speak call is now handled by the agent's final response usually
        try:
            # Assuming modified function takes TTS pipeline instance from SpeechManager
            return sceene_description_with_tts_modified(self.speech_manager.tts_pipeline, self.config.get('vision_llm_model_id'), self.config.get('lm_studio_base_url'))
        except Exception as e: error_msg = f"Error describing scene: {e}"; logger.exception(error_msg); self.speech_manager.speak(error_msg, interrupt=True); return error_msg

    def _execute_ocr(self) -> str:
        try:
            image_path = capture_image()
            if not image_path or not os.path.exists(image_path): raise FileNotFoundError("Failed image capture.")
            text = pytesseract.image_to_string(Image.open(image_path)); result_text = text.strip() or "No readable text found."
            self.speech_manager.speak(result_text, interrupt=True) # OCR speaks its direct result, maybe interrupt other speech
            return result_text
        except Exception as e: error_msg = f"Error reading text: {e}"; logger.exception(error_msg); self.speech_manager.speak(error_msg, interrupt=True); return error_msg

    def _execute_indoor_navigation_request(self, destination: str) -> str:
        """Handles the request to start navigation via the tool."""
        if not self.navigation_manager: return "Navigation system is unavailable."
        if self.navigation_manager.is_navigating: return "Already navigating. Cancel first."

        current_node_id = get_current_location_node_id() # Use global placeholder
        if not current_node_id: return "Cannot determine current location."

        destination_node_id = self.navigation_manager.node_map_manager.get_node_id_by_name(destination)
        if not destination_node_id: return f"Cannot find '{destination}' on map."

        if current_node_id == destination_node_id: return "Already at destination."

        current_head = get_current_heading() # Use global placeholder
        path = self.navigation_manager.node_map_manager.get_shortest_path(current_node_id, destination_node_id)
        if not path: return f"Cannot find path to {destination}."

        instructions = generate_navigation_instructions(path, self.navigation_manager.node_map_manager, current_head)
        if not instructions: return "Failed to generate instructions."

        # Start session using NavigationManager (which handles speech)
        if self.navigation_manager.start_session(path, instructions, destination):
            return f"Starting navigation to {destination}. First step: {instructions[0]}"
        else:
             return f"Failed to start navigation session to {destination}."

    def _execute_where_am_i(self) -> str:
         """Handles the 'Where am I?' request."""
         if self.navigation_manager and self.navigation_manager.is_navigating:
              status = self.navigation_manager.get_status()
         else:
              # Report nearest node/landmark based on current location
              current_node_id = get_current_location_node_id()
              if current_node_id and self.node_map_manager:
                   node_data = self.node_map_manager.get_node_data(current_node_id)
                   if node_data:
                       loc_name = node_data.get('name', f"node {current_node_id}")
                       loc_type = node_data.get('type')
                       landmark = node_data.get('landmark')
                       status = f"You are currently near {loc_name}"
                       if loc_type and loc_type not in ['Intersection', 'Area']: status += f" (a {loc_type})"
                       if landmark: status += f", close to {landmark}."
                       else: status += "."
                   else: status = "I know you're on the map, but I have no details for your current node."
              else: status = "I'm unable to determine your current location on the map."
         self.speech_manager.speak(status, interrupt=False) # Speak the location info
         return status # Return text for LLM context

    def _handle_command(self, command: str):
        """Processes a user command, interacting with LLM/Tools."""
        # --- Direct Command Handling (Before LLM) ---
        command_lower = command.lower()
        if command_lower == 'quit': return False # Signal to exit run loop
        if self.navigation_manager:
            if command_lower == 'cancel' or command_lower == 'cancel navigation':
                if not self.navigation_manager.cancel_session(): self.speech_manager.speak("Nothing to cancel.")
                return True # Command handled, don't invoke LLM
            elif command_lower == 'pause' or command_lower == 'pause navigation':
                 if not self.navigation_manager.pause_session(): self.speech_manager.speak("Not navigating or already paused.")
                 return True
            elif command_lower == 'resume' or command_lower == 'resume navigation':
                 if not self.navigation_manager.resume_session(): self.speech_manager.speak("Not paused or not navigating.")
                 return True
            elif command_lower == 'repeat' or command_lower == 'repeat instruction':
                 if not self.navigation_manager.repeat_instruction(): self.speech_manager.speak("Not navigating.")
                 return True
            elif command_lower == 'where am i': # Also handle via LLM tool for consistency? Or direct? Direct is faster.
                  self._execute_where_am_i()
                  return True # Command handled


        # --- LLM Agent Interaction ---
        try:
            self.chat_history.append(HumanMessage(content=command))
            logger.info("Invoking agent...")
            ai_response = self.llm_with_tools.invoke(self.chat_history)
            self.chat_history.append(ai_response)

            final_response_to_speak = ""

            if ai_response.tool_calls:
                logger.info(f"Agent wants to call tools: {ai_response.tool_calls}")
                tool_outputs = []
                for tool_call in ai_response.tool_calls:
                    tool_name = tool_call['name']; tool_args = tool_call['args']; tool_id = tool_call['id']
                    selected_tool = next((t for t in self.tools if t.name == tool_name), None)
                    if selected_tool:
                        try:
                            # Tool methods now return text, primary speech is inside them or via LLM final response
                            tool_result = selected_tool.invoke(tool_args)
                            tool_outputs.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))
                        except Exception as e: logger.exception(f"Error executing tool {tool_name}"); tool_outputs.append(ToolMessage(content=f"Error: {e}", tool_call_id=tool_id))
                    else: tool_outputs.append(ToolMessage(content=f"Error: Tool '{tool_name}' not found.", tool_call_id=tool_id))

                if tool_outputs:
                    self.chat_history.extend(tool_outputs)
                    logger.info("Sending tool results back to agent...")
                    final_ai_response = self.llm_with_tools.invoke(self.chat_history)
                    self.chat_history.append(final_ai_response)
                    final_response_to_speak = final_ai_response.content
            else: # No tool call
                final_response_to_speak = ai_response.content

            # Speak final LLM summary/response (if any)
            if final_response_to_speak:
                 self.speech_manager.speak(final_response_to_speak, interrupt=False) # Don't interrupt potential ongoing speech unless necessary

        except Exception as e:
            logger.exception("An error occurred during agent interaction.")
            self.speech_manager.speak("Sorry, I encountered an issue.", interrupt=True)
        finally:
             # Trim history
             max_hist = self.config.get('max_chat_history', 10)
             if len(self.chat_history) > max_hist: self.chat_history = self.chat_history[:1] + self.chat_history[-max_hist+1:]

        return True # Continue running

    def run(self):
        """Main interaction loop."""
        try:
            # Load TTS here after other components are ready
            self.speech_manager.load_tts()
        except Exception as e:
            logger.critical(f"Failed to load TTS. Cannot run assistant. Error: {e}", exc_info=True)
            return

        self.speech_manager.speak("Assistant ready.")
        input_thread = threading.Thread(target=command_input_thread, daemon=True)
        input_thread.start()

        last_command_check_time = time.time()

        keep_running = True
        while keep_running:
            current_time = time.time()

            # --- Navigation Update Logic ---
            if self.navigation_manager and self.navigation_manager.is_navigating:
                simulate_movement(self.navigation_manager.navigation_path) # Pass path for sim
                reroute_needed = self.navigation_manager.update_progress()
                if reroute_needed == "reroute":
                    # Trigger reroute - Need to get current loc *again* potentially
                    logger.info("Reroute requested by NavigationManager.")
                    current_node_id = get_current_location_node_id() # Get latest sim location
                    destination_node_id = self.navigation_manager.navigation_path[-1] # Keep same destination
                    current_head = get_current_heading()
                    # Cancel current session silently before starting new one
                    self.navigation_manager.cancel_session(speak_cancel=False)
                    # Use the internal execution method directly
                    reroute_result = self._execute_indoor_navigation_request(self.navigation_manager.navigation_destination_name) # Use name? or ID? Needs check
                    # Maybe speak the result explicitly?
                    self.speech_manager.speak(f"Recalculated route. {reroute_result}", interrupt=True)


            # --- Command Processing Logic ---
            command_interval = self.config.get('command_check_interval', 0.5)
            if current_time - last_command_check_time > command_interval:
                 command = listen_for_command_non_blocking()
                 last_command_check_time = current_time
                 if command:
                     keep_running = self._handle_command(command)


            # Prevent busy-waiting
            time.sleep(0.1)

        logger.info("Run loop finished.")


    def shutdown(self):
        """Cleanly shutdown resources."""
        logger.info("Shutting down Assistant.")
        if hasattr(self, 'speech_manager') and self.speech_manager:
             self.speech_manager.wait_for_speech_to_finish()
             self.speech_manager.shutdown()
        # Add any other cleanup needed (e.g., close sensor connections)
        logger.info("Assistant shutdown complete.")


# --- Main Execution ---
if __name__ == "__main__":
    assistant = None
    try:
        assistant = Assistant(config_path="config.yaml")
        assistant.run()
    except RuntimeError as e:
        logger.critical(f"Initialization failed: {e}", exc_info=True)
    except FileNotFoundError as e:
         logger.critical(f"Critical Error: {e}. Please ensure map.json and config.yaml exist.", exc_info=True)
    except Exception as e:
        logger.critical(f"An unexpected error occurred: {e}", exc_info=True)
    finally:
        if assistant:
            assistant.shutdown()
        logger.info("Application exiting.")

In [1]:
# Standard Library Imports
import json
import math
import os
import threading
import queue
import time
import re
import logging
from typing import Optional, List, Dict, Any, Tuple
import base64 # Needed for image encoding

# Third-party Imports
import networkx as nx
try:
    import sounddevice as sd
    SOUNDDEVICE_AVAILABLE = True
except (ImportError, OSError) as e:
    logging.warning(f"Sounddevice import failed ({e}). TTS playback will be disabled. Install 'sounddevice' and potentially 'portaudio'.")
    SOUNDDEVICE_AVAILABLE = False
# import numpy as np # Keep if TTS model output requires it
from PIL import Image
import pytesseract
import yaml # For config file
import cv2 # Added for image capture example

# LangChain Imports
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, AIMessageChunk # Added AIMessageChunk if streaming
from langchain_openai import ChatOpenAI

# Your existing utils (assuming they exist and are correct)
# IMPORTANT: Ensure these utils are compatible with changes if any
# from utils.model import initiate_llm, initiate_tts_model # Assume these work
# from utils.to_base64 import encode_image_to_base64 # We implement a version here
# from utils.audio_player import audio_player # Replaced by direct sounddevice use
# from utils.audio_generator import audio_generator # Replaced by direct TTS use
# from utils.snap_a_picture import capture_image # Implemented placeholder below

# --- Basic Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Placeholder/Example Implementations ---

def initiate_tts_model(desired_device='cpu'):
    """Placeholder: Load your actual TTS model pipeline (e.g., Transformers, Piper)."""
    logger.warning("Using PLACEHOLDER TTS model initializer.")
    # Example with a hypothetical simple interface
    try:
        # Replace with your actual TTS loading logic
        # from transformers import pipeline
        # tts_pipeline = pipeline("text-to-speech", model="your-tts-model-id", device=desired_device)
        # return tts_pipeline
        # Mock implementation for testing structure
        class MockTTSPipeline:
            def __init__(self, device):
                logger.info(f"Mock TTS initialized on device: {device}")
                # For numpy usage if needed:
                # import numpy as np
                # self.np = np

            def __call__(self, text: str, **kwargs) -> Dict[str, Any]:
                logger.info(f"Mock TTS generating audio for: '{text[:30]}...'")
                # Simulate audio generation
                # Return format might depend on your TTS lib (e.g., dict with 'audio', 'sampling_rate')
                # Using a simple placeholder - in reality, this would be raw audio data (like a numpy array)
                # Replace with actual TTS call and result handling
                sample_rate = 24000 # Example
                # audio_data = self.np.random.uniform(-0.5, 0.5, size=int(len(text) * 0.2 * sample_rate)) # Dummy audio
                audio_data = b'\x00' * int(len(text) * 0.2 * sample_rate) # Dummy silent audio bytes for structure testing
                return {"audio": audio_data, "sampling_rate": sample_rate} # Or just the audio data if simpler
        return MockTTSPipeline(desired_device)
    except Exception as e:
        logger.exception(f"Failed to initialize placeholder TTS model: {e}")
        return None

def encode_image_to_base64(image_path):
    """Encodes an image file to a base64 string."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        logger.error(f"Image file not found for encoding: {image_path}")
        return None
    except Exception as e:
        logger.exception(f"Error encoding image {image_path} to base64")
        return None

def capture_image(filepath="captured_image.jpg") -> Optional[str]:
    """
    Captures an image using the default camera and saves it.
    Returns the filepath if successful, None otherwise.
    Requires opencv-python installed.
    """
    logger.info("Attempting to capture image...")
    try:
        # Initialize the camera (0 is usually the default camera)
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            logger.error("Cannot open camera")
            return None

        # Allow camera to warm up
        time.sleep(0.5)

        # Capture a single frame
        ret, frame = cap.read()

        # Release the camera
        cap.release()

        if ret:
            # Save the frame as an image file
            cv2.imwrite(filepath, frame)
            logger.info(f"Image captured successfully and saved to {filepath}")
            return filepath
        else:
            logger.error("Failed to capture frame from camera")
            return None
    except ImportError:
        logger.error("OpenCV (cv2) is not installed. Cannot capture image. Please install opencv-python or opencv-python-headless.")
        # Fallback: return a path to a dummy image if it exists
        dummy_path = "placeholder_image.jpg"
        if os.path.exists(dummy_path):
             logger.warning(f"Using placeholder image: {dummy_path}")
             return dummy_path
        else:
             logger.error("No placeholder image found.")
             return None
    except Exception as e:
        logger.exception(f"An error occurred during image capture: {e}")
        return None

# --- NodeMapManager Class (Keep as defined previously, minor logging) ---
# (Assuming the provided NodeMapManager class is functionally correct)
class NodeMapManager:
    def __init__(self, map_filepath):
        logger.info(f"Initializing NodeMapManager with map: {map_filepath}")
        if not os.path.exists(map_filepath):
             logger.error(f"Map file not found: {map_filepath}")
             raise FileNotFoundError(f"Map file not found: {map_filepath}")
        self.map_data = self._load_map(map_filepath)
        self.graph = nx.DiGraph()
        self._build_graph()
        self._node_name_to_id = {
            str(node.get('name', '')).lower(): str(node['id'])
            for node in self.map_data.get('nodes', []) if 'name' in node and 'id' in node
        }
        self._node_id_to_data = { # Cache for faster lookups
             str(node['id']): node for node in self.map_data.get('nodes', []) if 'id' in node
        }
        logger.info(f"Loaded map with {self.graph.number_of_nodes()} nodes and {self.graph.number_of_edges()} edges.")

    def _load_map(self, filepath):
        try:
            with open(filepath, 'r') as f:
                map_data = json.load(f)
            # Basic Validation
            if 'nodes' not in map_data or not isinstance(map_data['nodes'], list):
                 raise ValueError("Map data must contain a 'nodes' list.")
            return map_data
        except json.JSONDecodeError as e:
             logger.error(f"Invalid JSON in map file {filepath}: {e}")
             raise
        except Exception as e:
            logger.exception(f"Failed to load or parse map file {filepath}")
            raise

    def get_shortest_path(self, start_node_id, end_node_id):
        try:
            # Ensure IDs are strings for graph lookup
            s_id = str(start_node_id)
            e_id = str(end_node_id)
            if not self.graph.has_node(s_id): raise nx.NodeNotFound(f"Start node {s_id} not in graph")
            if not self.graph.has_node(e_id): raise nx.NodeNotFound(f"End node {e_id} not in graph")

            path = nx.shortest_path(self.graph, source=s_id, target=e_id, weight='weight')
            logger.debug(f"Shortest path found: {' -> '.join(path)}")
            return path
        except nx.NetworkXNoPath:
            logger.warning(f"No path found between {start_node_id} and {end_node_id}")
            return None
        except nx.NodeNotFound as e:
            logger.warning(f"Node not found in graph during pathfinding: {e}")
            return None
        except Exception as e:
            logger.exception("Error during pathfinding")
            return None

    def get_node_data(self, node_id) -> Optional[Dict[str, Any]]:
        # Use cached lookup
        return self._node_id_to_data.get(str(node_id))
        # try:
        #     return self.graph.nodes[str(node_id)] # networkx stores copy, lookup might be slower
        # except KeyError:
        #     logger.debug(f"Node data requested for unknown ID: {node_id}") # Less noisy log
        #     return None

    def get_node_id_by_name(self, name: str) -> Optional[str]:
         return self._node_name_to_id.get(name.lower())

    def _build_graph(self):
        if 'nodes' not in self.map_data:
            logger.error("Map data is missing 'nodes' key. Cannot build graph.")
            return
        for node_data in self.map_data['nodes']:
             node_id = str(node_data.get('id', None))
             if node_id is None:
                 logger.warning(f"Skipping node with missing ID: {node_data.get('name','Unnamed')}")
                 continue
             # Ensure x, y exist and are numbers, default to 0
             node_attrs = node_data.copy()
             try: node_attrs['x'] = float(node_attrs.get('x', 0.0))
             except (ValueError, TypeError): node_attrs['x'] = 0.0; logger.warning(f"Node {node_id} has invalid 'x', defaulting to 0.")
             try: node_attrs['y'] = float(node_attrs.get('y', 0.0))
             except (ValueError, TypeError): node_attrs['y'] = 0.0; logger.warning(f"Node {node_id} has invalid 'y', defaulting to 0.")

             self.graph.add_node(node_id, **node_attrs)

        # Add edges based on 'links' and 'outgoingLinks'
        edges_added = set() # To avoid duplicate edges if defined in both places

        # Process top-level 'links' first (often used for undirected/reciprocal links)
        if 'links' in self.map_data and isinstance(self.map_data['links'], list):
            for link in self.map_data['links']:
                s = str(link.get('startNode'))
                e = str(link.get('endNode'))
                if self.graph.has_node(s) and self.graph.has_node(e):
                    dist = self.calculate_distance_by_ids(s, e)
                    if dist != float('inf'):
                        # Assume bidirectional unless otherwise specified
                        if (s, e) not in edges_added: self.graph.add_edge(s, e, weight=dist); edges_added.add((s, e))
                        if (e, s) not in edges_added: self.graph.add_edge(e, s, weight=dist); edges_added.add((e, s))
                else:
                     logger.warning(f"Skipping link due to missing node(s): {s} <-> {e}")

        # Process 'outgoingLinks' within each node (for directed links)
        for node_data in self.map_data['nodes']:
            start_node_id = str(node_data.get('id'))
            if not self.graph.has_node(start_node_id): continue # Skip if start node wasn't added

            if 'outgoingLinks' in node_data and isinstance(node_data['outgoingLinks'], list):
                for link in node_data['outgoingLinks']:
                    end_node_id = str(link.get('endNode'))
                    if self.graph.has_node(end_node_id):
                        if (start_node_id, end_node_id) not in edges_added: # Only add if not covered by 'links'
                            dist = self.calculate_distance_by_ids(start_node_id, end_node_id)
                            if dist != float('inf'):
                                self.graph.add_edge(start_node_id, end_node_id, weight=dist)
                                edges_added.add((start_node_id, end_node_id))
                                # Optional: Add reverse link if map implies bidirectionality?
                                # if (end_node_id, start_node_id) not in edges_added:
                                #     self.graph.add_edge(end_node_id, start_node_id, weight=dist)
                                #     edges_added.add((end_node_id, start_node_id))
                        # Else: Link already exists, potentially from 'links' section.
                    else:
                         logger.warning(f"Skipping outgoingLink from {start_node_id} to missing node {end_node_id}")

    def calculate_distance(self, node1_data, node2_data):
        # Assumes 'x' and 'y' keys exist and are numeric after _build_graph ensures it
        x1=node1_data['x']; y1=node1_data['y']; x2=node2_data['x']; y2=node2_data['y']
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    def calculate_distance_by_ids(self, node1_id, node2_id):
         node1_data = self.get_node_data(node1_id); node2_data = self.get_node_data(node2_id)
         if node1_data and node2_data: return self.calculate_distance(node1_data, node2_data)
         else: return float('inf') # Return infinity if nodes don't exist

# --- Simulation State & Placeholders (Improved for Pause/Resume) ---
SIMULATED_USER_NODE_ID = "node1" # Default start node ID (make configurable?)
SIMULATED_USER_HEADING = 0.0 # Degrees, 0=East, 90=North, 180=West, 270=South
SIMULATED_MOVEMENT_PAUSED = False # Flag for pause testing
_sim_last_update_time = time.time()
_sim_target_node_id = None
_sim_movement_delay = 3.0 # Time to "reach" next node (make configurable?)

def update_simulation_target(target_node_id: Optional[str]):
    global _sim_target_node_id, _sim_last_update_time
    if _sim_target_node_id != target_node_id:
        logger.debug(f"SIM: New movement target set to {target_node_id}")
        _sim_target_node_id = target_node_id
        _sim_last_update_time = time.time() # Reset timer when target changes

def simulate_movement(current_path: Optional[List[str]] = None, node_map_manager: Optional[NodeMapManager] = None):
    """Simulates user moving to the _sim_target_node_id after a delay."""
    global SIMULATED_USER_NODE_ID, SIMULATED_USER_HEADING, _sim_last_update_time, _sim_target_node_id
    if SIMULATED_MOVEMENT_PAUSED or not _sim_target_node_id:
        return # Don't move if paused or no target

    current_time = time.time()
    if current_time - _sim_last_update_time >= _sim_movement_delay:
        logger.info(f"SIM: Simulating arrival at {_sim_target_node_id} from {SIMULATED_USER_NODE_ID}")

        # Update heading *before* updating the node ID
        # Calculate heading from the node we just arrived at (_sim_target_node_id)
        # towards the *next* node in the path (if available).
        if current_path and node_map_manager:
             try:
                  # Find the index of the node we are arriving at
                  current_index = current_path.index(_sim_target_node_id)
                  if current_index + 1 < len(current_path):
                      next_node_on_path_id = current_path[current_index + 1]
                      arrived_node_data = node_map_manager.get_node_data(_sim_target_node_id)
                      next_node_data = node_map_manager.get_node_data(next_node_on_path_id)
                      if arrived_node_data and next_node_data:
                           new_heading = get_vector_heading(
                               arrived_node_data['x'], arrived_node_data['y'],
                               next_node_data['x'], next_node_data['y']
                           )
                           logger.debug(f"SIM: Updating heading from {SIMULATED_USER_HEADING:.1f} to {new_heading:.1f} (towards {next_node_on_path_id})")
                           SIMULATED_USER_HEADING = new_heading
                      else: logger.warning("SIM: Could not get node data for heading update.")
                  # else: Arrived at final destination, heading doesn't matter as much
             except (ValueError, IndexError):
                  logger.warning(f"SIM: Could not find target node {_sim_target_node_id} in path for heading update.")
        else:
             logger.debug("SIM: Cannot update heading without path or map manager.")


        # Arrived! Update current node ID
        SIMULATED_USER_NODE_ID = _sim_target_node_id
        logger.info(f"SIM: User is now at node {SIMULATED_USER_NODE_ID}")
        update_simulation_target(None) # Clear target, wait for next instruction/step

# --- Localization Interface ---
# In this version, these rely purely on the simulation state.
# A real system would replace these with sensor fusion / VPR logic.
def get_current_location_node_id() -> Optional[str]:
    """Returns the globally simulated user node ID."""
    # In a real system: Query the localization module (sensor fusion, VPR etc.)
    # logger.debug(f"Localization Query: Returning simulated node {SIMULATED_USER_NODE_ID}")
    return SIMULATED_USER_NODE_ID

def get_current_heading() -> float:
    """Returns the globally simulated user heading (degrees, 0=East, 90=North)."""
    # In a real system: Query the localization module (IMU/Compass fusion)
    # logger.debug(f"Localization Query: Returning simulated heading {SIMULATED_USER_HEADING:.1f}")
    return SIMULATED_USER_HEADING

# --- STT Placeholder (Keep as before) ---
_command_queue = queue.Queue()
def listen_for_command_non_blocking():
    if not _command_queue.empty(): return _command_queue.get()
    return None
def command_input_thread():
    time.sleep(5) # Allow startup messages
    print("\n----- Enter commands ('nav [destination]', 'describe', 'read', 'pause', 'resume', 'where am i', 'repeat', 'cancel', 'quit') -----")
    while True:
        try: command = input("> "); _command_queue.put(command)
        except EOFError: _command_queue.put('quit'); break # Handle Ctrl+D
        except KeyboardInterrupt: _command_queue.put('quit'); break # Handle Ctrl+C
        if command.lower() == 'quit': break

# --- Navigation Instruction Generation (Keep as defined previously, ensure functions exist) ---
# calculate_angle_and_turn, get_relative_turn_command, get_vector_heading, generate_navigation_instructions
# (Make sure these are defined correctly as in previous steps or included here)
def calculate_angle_and_turn(current_heading: float, target_heading: float) -> float:
    """Calculates the shortest angle difference between two headings (degrees)."""
    current_heading %= 360
    target_heading %= 360
    angle_diff = target_heading - current_heading
    # Normalize to -180 to 180 range
    if angle_diff > 180:
        angle_diff -= 360
    elif angle_diff <= -180:
        angle_diff += 360
    return angle_diff

def get_relative_turn_command(turn_angle: float) -> str:
    """Converts a turn angle (degrees) into a human-readable command."""
    abs_angle = abs(turn_angle)
    direction = "right" if turn_angle > 0 else "left"

    if abs_angle < 15: # Threshold for "straight"
        return "Continue straight"
    elif abs_angle < 60:
        return f"Turn slightly {direction}"
    elif abs_angle < 120:
        return f"Turn {direction}"
    elif abs_angle < 165: # Threshold for "sharp" vs "U-turn"
        return f"Turn sharply {direction}"
    else:
        return "Make a U-turn" # Approaching 180 degrees

def get_vector_heading(x1: float, y1: float, x2: float, y2: float) -> float:
    """Calculates the heading (degrees) of the vector from (x1, y1) to (x2, y2).
       0=East, 90=North, 180=West, 270=South. Matches simulation convention.
    """
    angle_rad = math.atan2(y2 - y1, x2 - x1)
    angle_deg = math.degrees(angle_rad)
    return angle_deg % 360 # Normalize to 0-360

def generate_navigation_instructions(path: List[str], node_map_manager: NodeMapManager, initial_heading: float) -> List[str]:
    """Generates turn-by-turn instructions from a path of node IDs."""
    if not path or len(path) < 2 or not node_map_manager:
        return []

    instructions = []
    current_heading = initial_heading

    for i in range(len(path) - 1):
        current_node_id = path[i]
        next_node_id = path[i+1]
        next_next_node_id = path[i+2] if i + 2 < len(path) else None

        current_node_data = node_map_manager.get_node_data(current_node_id)
        next_node_data = node_map_manager.get_node_data(next_node_id)

        if not current_node_data or not next_node_data:
            logger.error(f"Cannot generate instruction step: Missing data for nodes {current_node_id} or {next_node_id}")
            instructions.append(f"Error: Could not process step from {current_node_id} to {next_node_id}.")
            # Attempt to recover heading if possible, otherwise use last known
            # This is complex; for now, we might lose accurate heading after an error.
            continue

        # 1. Calculate heading needed to reach the *next* node
        target_heading = get_vector_heading(current_node_data['x'], current_node_data['y'],
                                            next_node_data['x'], next_node_data['y'])

        # 2. Calculate distance to the *next* node
        distance = node_map_manager.calculate_distance(current_node_data, next_node_data)

        # 3. Calculate turn required from *current* heading to *target* heading
        turn_angle = calculate_angle_and_turn(current_heading, target_heading)
        turn_command = get_relative_turn_command(turn_angle)

        # 4. Construct the instruction string
        step_instruction = ""
        next_node_name = next_node_data.get('name', f'node {next_node_id}') # Use name if available

        # Add turn command (unless it's the very first step and turn is minor)
        is_first_step = (i == 0)
        if not (is_first_step and turn_command == "Continue straight"):
             if turn_command == "Continue straight":
                 # Be less verbose if already going straight
                 if not instructions or "straight" not in instructions[-1].lower():
                     step_instruction += f"{turn_command}. "
             else:
                 step_instruction += f"{turn_command}. "


        # Add action phrase (Proceed/Head towards)
        if is_first_step:
            step_instruction += f"Head towards {next_node_name}, "
        else:
            # Check if previous instruction already mentioned the target
             if not instructions or next_node_name not in instructions[-1]:
                 step_instruction += f"Proceed towards {next_node_name}, "
             else: # Avoid redundancy "Turn right towards X. Proceed towards X" -> "Turn right towards X."
                 step_instruction += "Proceed "


        step_instruction += f"for about {distance:.1f} meters."

        # Add landmark/type info for the *next* node (the one we're heading towards)
        landmark = next_node_data.get('landmark')
        node_type = next_node_data.get('type')
        if landmark:
             step_instruction += f" You should pass {landmark}."
        elif node_type and node_type not in ['Intersection', 'Area', '']: # Don't mention generic types
             step_instruction += f" It's near the {node_type}."


        # 5. Add "prepare to turn" hint for the *following* step, if applicable
        if next_next_node_id:
            next_next_node_data = node_map_manager.get_node_data(next_next_node_id)
            if next_next_node_data:
                # Heading required from 'next_node' to 'next_next_node'
                next_segment_heading = get_vector_heading(next_node_data['x'], next_node_data['y'],
                                                          next_next_node_data['x'], next_next_node_data['y'])
                # Turn required *at* the next node (from target_heading to next_segment_heading)
                upcoming_turn_angle = calculate_angle_and_turn(target_heading, next_segment_heading)
                upcoming_turn_command = get_relative_turn_command(upcoming_turn_angle)

                if upcoming_turn_command != "Continue straight":
                    step_instruction += f" Then prepare to {upcoming_turn_command.lower()}."

        instructions.append(step_instruction.strip())

        # Update current heading for the next iteration *after* calculations for this step
        current_heading = target_heading

    # Add arrival message
    destination_node_data = node_map_manager.get_node_data(path[-1])
    if destination_node_data:
        dest_name = destination_node_data.get('name', path[-1])
        dest_type = destination_node_data.get('type')
        arrival_msg = f"You have arrived at {dest_name}"
        if dest_type and dest_type not in ['Intersection', 'Area', '']:
            arrival_msg += f" (the {dest_type})."
        else:
            arrival_msg += "."
        instructions.append(arrival_msg)
    else:
        instructions.append("You have arrived at your destination.")

    return instructions


# --- Vision Function (using LLM) ---
def sceene_description_with_tts_modified(vision_llm: ChatOpenAI, image_path: Optional[str]) -> str:
    """
    Gets scene description from a vision LLM for a captured image.
    TTS is handled separately by the caller (Assistant -> SpeechManager).
    """
    if not image_path:
        return "Could not capture an image to describe."

    logger.info(f"Getting scene description for image: {image_path}")
    base64_image = encode_image_to_base64(image_path)
    if not base64_image:
        return "Could not encode image for analysis."

    prompt = """Describe the scene in this image focusing on potential obstacles, pathways, landmarks, and overall layout for a visually impaired user. Be concise and informative."""
    # Construct message compatible with vision models (e.g., OpenAI format)
    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{base64_image}",
            },
        ]
    )

    try:
        response = vision_llm.invoke([message])
        description = response.content if hasattr(response, 'content') else str(response)
        logger.info(f"Scene description received: '{description[:100]}...'")
        return description.strip() if description else "Could not get a description from the vision model."
    except Exception as e:
        logger.exception("Error invoking vision LLM for scene description")
        return f"Error getting scene description: {e}"


# --- Navigation Manager (Mostly unchanged, relies on simulation) ---
class NavigationManager:
    def __init__(self, assistant_speak_func, node_map_manager, config):
        self.speak = assistant_speak_func # Function to speak text
        self.node_map_manager = node_map_manager
        self.config = config
        self.is_navigating = False
        self.is_paused = False
        self.navigation_path: List[str] = []
        self.navigation_instructions: List[str] = []
        self.current_nav_step: int = 0
        self.navigation_destination_name: str = ""
        self.navigation_destination_id: str = "" # Store ID too
        self.last_nav_update_time: float = 0
        self.off_route_counter: int = 0 # Count consecutive off-route checks

    def start_session(self, path, instructions, destination_name, destination_id):
        if not path or not instructions:
            logger.error("Navigation session start failed: Missing path or instructions.")
            return False
        if len(path) < 2:
             logger.error("Navigation session start failed: Path requires at least two nodes.")
             return False
        self.navigation_path = path
        self.navigation_instructions = instructions
        self.current_nav_step = 0
        self.is_navigating = True
        self.is_paused = False
        self.navigation_destination_name = destination_name
        self.navigation_destination_id = destination_id
        self.last_nav_update_time = time.time()
        self.off_route_counter = 0

        logger.info(f"Starting navigation to {destination_name} ({destination_id}). Path: {'->'.join(path)}")

        # Update simulation target for the first step's *target* node
        first_target_node = self.navigation_path[1] # Path has at least 2 nodes ensured above
        update_simulation_target(first_target_node)

        # Speak the first instruction
        # Use interrupt=True for the *first* instruction to ensure it's heard clearly.
        self.speak(self.navigation_instructions[self.current_nav_step], interrupt=True)
        return True

    def cancel_session(self, speak_cancel=True):
        if self.is_navigating:
            logger.info("Cancelling navigation.")
            was_paused = self.is_paused
            self.is_navigating = False
            self.is_paused = False
            update_simulation_target(None) # Stop simulation movement
            if not was_paused: # Resume sim movement if it was paused systemically
                 global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = False
            if speak_cancel: self.speak("Navigation cancelled.")
            # Reset state fully
            self.navigation_path = []; self.navigation_instructions = []; self.current_nav_step = 0; self.navigation_destination_name = ""; self.navigation_destination_id = ""
            return True
        return False

    def pause_session(self):
        if self.is_navigating and not self.is_paused:
            logger.info("Pausing navigation.")
            self.is_paused = True
            global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = True # Pause simulation
            self.speak("Navigation paused.")
            return True
        elif self.is_navigating and self.is_paused:
             logger.info("Navigation already paused.")
             self.speak("Navigation is already paused.")
             return True # Or False depending on desired behavior for re-pausing
        return False # Not navigating


    def resume_session(self):
        if self.is_navigating and self.is_paused:
            logger.info("Resuming navigation.")
            self.is_paused = False
            global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = False # Resume simulation
            self.last_nav_update_time = time.time() # Reset update timer

            # Re-speak current instruction and ensure simulation target is set
            self.speak(f"Resuming navigation. Current step: {self.navigation_instructions[self.current_nav_step]}", interrupt=True) # Interrupt ensures it's heard

            # Re-set simulation target based on current step
            if self.current_nav_step + 1 < len(self.navigation_path):
                 next_target_node = self.navigation_path[self.current_nav_step + 1]
                 update_simulation_target(next_target_node)
            # else: we are on the arrival instruction, no target needed
            return True
        elif self.is_navigating and not self.is_paused:
             logger.info("Navigation is not paused.")
             self.speak("Navigation is already active.")
             return True # Or False
        return False # Not navigating

    def get_status(self):
        """Reports current navigation status."""
        if not self.is_navigating:
            return "Not currently navigating."

        status = f"Navigating towards {self.navigation_destination_name}. "
        if self.is_paused:
            status += "Status: Paused. "
        else:
            status += "Status: Active. "

        if self.current_nav_step < len(self.navigation_instructions):
            status += f"Current instruction: {self.navigation_instructions[self.current_nav_step]}"
        else:
            status += "Processing arrival..." # Should ideally not happen if arrival clears state

        # TODO: Add distance/ETA estimate? Requires tracking progress along segment.
        return status

    def repeat_instruction(self):
         if self.is_navigating and self.current_nav_step < len(self.navigation_instructions):
              logger.info(f"Repeating instruction: {self.navigation_instructions[self.current_nav_step]}")
              # Interrupt any previous speech fragment to ensure the instruction is clear.
              self.speak(f"Current instruction: {self.navigation_instructions[self.current_nav_step]}", interrupt=True)
              return True
         elif self.is_navigating:
             logger.warning("Repeat requested but step index out of bounds.")
             return False # Should not happen
         return False # Not navigating

    def update_progress(self) -> Optional[str]:
        """Checks user location (simulation), advances steps, or triggers reroute. Returns 'reroute' if needed."""
        if not self.is_navigating or self.is_paused or not self.node_map_manager:
            return None

        current_time = time.time()
        # Use configured update interval
        nav_update_interval = self.config.get('navigation_update_interval', 2.0)
        if current_time - self.last_nav_update_time < nav_update_interval:
             return None # Don't check too frequently

        self.last_nav_update_time = current_time
        current_loc_id = get_current_location_node_id() # From simulation

        if not current_loc_id:
            logger.warning("Nav Update: Could not get current location (simulation error?).")
            # Maybe speak "Lost tracking?" after several failures?
            return None

        # --- Check if we are on the expected path segment ---
        if self.current_nav_step >= len(self.navigation_path) - 1:
             # We are on the last step (arrival instruction), waiting for simulation potentially?
             # Or maybe arrival already happened in _advance_step.
             # logger.debug("Nav Update: On arrival instruction, no further progress check needed.")
             # Let's double check if the user *is* at the destination node
             if current_loc_id == self.navigation_destination_id:
                 logger.info("Nav Update: Confirmed arrival at destination.")
                 # If somehow is_navigating is still true, force it false.
                 if self.is_navigating: self._handle_arrival()
             return None

        # Expected nodes for the current step
        current_expected_node = self.navigation_path[self.current_nav_step]
        target_node_id = self.navigation_path[self.current_nav_step + 1]

        # --- Scenario 1: User has reached the target node for the current step ---
        if current_loc_id == target_node_id:
            logger.info(f"Nav Update: User reached target {target_node_id} for step {self.current_nav_step}.")
            self._advance_step()
            self.off_route_counter = 0 # Reset counter on successful step
            return None # Handled internally

        # --- Scenario 2: User is *not* at the target node yet ---
        # Check if the user is still at the *expected starting node* for this segment
        elif current_loc_id == current_expected_node:
            # User is on the correct segment, but hasn't reached the target yet. Normal operation.
            self.off_route_counter = 0 # Reset counter if back on the expected node
            logger.debug(f"Nav Update: User still at {current_loc_id}, proceeding towards {target_node_id}.")
            # Ensure simulation target is still set correctly
            if _sim_target_node_id != target_node_id: update_simulation_target(target_node_id)
            return None

        # --- Scenario 3: User is at an unexpected node (Off-Route) ---
        else:
            self.off_route_counter += 1
            logger.warning(f"Nav Update: User at UNEXPECTED node {current_loc_id}. Expected segment {current_expected_node} -> {target_node_id}. (Off-route count: {self.off_route_counter})")
            # Trigger reroute check if counter exceeds threshold
            off_route_threshold = self.config.get('off_route_threshold_nodes', 2)
            if self.off_route_counter >= off_route_threshold:
                 logger.warning(f"Triggering reroute. Off-route count ({self.off_route_counter}) meets threshold ({off_route_threshold}).")
                 # Speak message handled by Assistant before calling reroute tool/function
                 # self.speak("It seems you are off the planned route. Let me recalculate.")
                 return "reroute" # Signal to Assistant to reroute
            else:
                 # Still potentially off-route, but haven't hit threshold. Maybe speak a warning?
                 # self.speak("Are you still on the path?") # Could be annoying
                 pass
            return None

    def _advance_step(self):
        """Internal: Moves to next step, speaks instruction or handles arrival."""
        self.current_nav_step += 1
        if self.current_nav_step < len(self.navigation_instructions):
            instruction = self.navigation_instructions[self.current_nav_step]
            is_arrival_instruction = (self.current_nav_step == len(self.navigation_instructions) - 1)

            logger.info(f"Advancing to Nav Step {self.current_nav_step}: {instruction}")
            # Use interrupt for critical turns/arrival? Maybe only arrival.
            self.speak(instruction, interrupt=is_arrival_instruction)

            if is_arrival_instruction:
                self._handle_arrival()
            else:
                 # Update simulation target for the *next* step's target node
                 if self.current_nav_step + 1 < len(self.navigation_path):
                      next_target_node = self.navigation_path[self.current_nav_step + 1]
                      update_simulation_target(next_target_node)
                 else:
                      logger.error("Logic Error: Should have been arrival instruction but path index out of bounds.")
                      self._handle_arrival() # Treat as arrival anyway
        else:
            # This case should ideally be handled by the arrival check before incrementing. Safety fallback.
            logger.warning("Tried to advance step but already at/past the last instruction index.")
            if self.is_navigating: # If somehow still navigating, force arrival state
                 self._handle_arrival()

    def _handle_arrival(self):
        """Internal: Cleans up state upon reaching the destination."""
        logger.info("Navigation complete. Reached destination.")
        # Speak final confirmation (might be redundant if last instruction is arrival msg)
        # self.speak("Navigation complete.", interrupt=True) # Already spoken by _advance_step usually
        self.is_navigating = False
        self.is_paused = False
        # Reset state *after* potential reroute logic might need destination
        # self.navigation_path = []; self.navigation_instructions = []; self.current_nav_step = 0; self.navigation_destination_name = ""; self.navigation_destination_id = ""
        # Clear simulation target explicitly
        update_simulation_target(None)
        global SIMULATED_MOVEMENT_PAUSED; SIMULATED_MOVEMENT_PAUSED = False # Ensure simulation can move if paused


# --- Speech Manager (Revised for direct TTS/Playback) ---
class SpeechManager:
    def __init__(self, config):
        self.config = config
        self.tts_pipeline = None # Loaded later
        self.speak_queue = queue.Queue()
        self.speak_thread = threading.Thread(target=self._speech_worker, daemon=True)
        self.stop_event = threading.Event()
        self.interrupt_event = threading.Event() # Event to signal interruption

    def load_tts(self):
        """Loads the TTS model."""
        if self.tts_pipeline: return # Already loaded
        logger.info("Initializing TTS in SpeechManager...")
        try:
            # Ensure initiate_tts_model exists and works
            self.tts_pipeline = initiate_tts_model(self.config.get('tts_device', 'cpu'))
            if not self.tts_pipeline:
                 raise RuntimeError("initiate_tts_model returned None.")
            if not SOUNDDEVICE_AVAILABLE:
                 logger.error("Sounddevice not available. Cannot start speech worker.")
                 raise RuntimeError("Sounddevice unavailable.")
            self.speak_thread.start()
            logger.info("TTS loaded and speech worker started.")
        except Exception as e:
            logger.exception("Failed to initialize TTS pipeline or start worker in SpeechManager.")
            # Ensure TTS pipeline is None if loading failed
            self.tts_pipeline = None
            # Do not raise here, allow Assistant to potentially continue without speech
            # raise RuntimeError(f"TTS failed to load: {e}")

    def speak(self, text: str, interrupt: bool = False):
        """Add text to the speech queue. If interrupt, clear queue and signal worker."""
        if not self.tts_pipeline or not SOUNDDEVICE_AVAILABLE:
            logger.warning(f"Speech skipped (TTS unavailable): '{text[:50]}...'")
            return
        if not text: return

        if interrupt:
            logger.debug("Interrupting speech queue and current playback.")
            self._clear_queue()
            self.interrupt_event.set() # Signal the worker to stop current playback

        logger.debug(f"Queuing speech: '{text[:50]}...'")
        self.speak_queue.put(text)

    def _clear_queue(self):
        """Clears the pending speech queue."""
        while not self.speak_queue.empty():
            try: self.speak_queue.get_nowait()
            except queue.Empty: break

    def _speech_worker(self):
        """Worker thread that processes the speech queue using TTS and sounddevice."""
        while not self.stop_event.is_set():
            try:
                # Wait for text, check for stop/interrupt periodically
                try:
                    text_to_speak = self.speak_queue.get(timeout=0.2)
                except queue.Empty:
                    if self.interrupt_event.is_set(): self.interrupt_event.clear() # Clear interrupt if queue is empty
                    continue # No text, loop again

                if text_to_speak is None: # Sentinel for shutdown
                     break

                logger.info(f"Speech Worker: Processing '{text_to_speak[:50]}...'")

                # 1. Generate Audio using the TTS pipeline
                try:
                    # Ensure tts_pipeline call matches its expected signature and return format
                    # Example assumes it returns a dict with 'audio' (numpy array or bytes) and 'sampling_rate'
                    # Or it might return just the audio data directly
                    tts_output = self.tts_pipeline(text_to_speak, voice=self.config.get('tts_voice','am_onyx')) # Pass config params

                    # Extract audio data and sample rate - adjust based on actual TTS lib
                    if isinstance(tts_output, dict):
                        audio_data = tts_output.get('audio')
                        sample_rate = tts_output.get('sampling_rate')
                    else: # Assume raw audio data if not dict
                        audio_data = tts_output
                        sample_rate = self.config.get('audio_sample_rate', 24000) # Get rate from config

                    if audio_data is None or sample_rate is None:
                         raise ValueError("TTS pipeline did not return valid audio data or sample rate.")
                     # Convert to numpy array if needed by sounddevice and not already
                     # if not isinstance(audio_data, np.ndarray):
                     #     # Need to know the format (e.g., bytes, int16, float32)
                     #     pass # Conversion logic here if necessary

                except Exception as e:
                    logger.exception(f"Speech Worker: Error during TTS generation for '{text_to_speak[:50]}...'")
                    self.speak_queue.task_done() # Mark task done even if TTS failed
                    continue # Skip to next item

                # 2. Play Audio using sounddevice
                try:
                    logger.debug(f"Speech Worker: Playing audio ({type(audio_data)}, Sample Rate: {sample_rate})...")
                    # Play the audio data. sd.play is non-blocking by default.
                    sd.play(audio_data, sample_rate, blocking=False)

                    # Wait for playback to complete OR for an interrupt signal
                    while sd.get_stream().active:
                        if self.interrupt_event.is_set() or self.stop_event.is_set():
                            logger.info("Speech Worker: Interrupt detected, stopping playback.")
                            sd.stop()
                            self.interrupt_event.clear() # Reset interrupt flag
                            break
                        time.sleep(0.05) # Short sleep to avoid busy-waiting

                    # Ensure stream is stopped if loop exited normally (playback finished)
                    # sd.stop() # Might cut off last bit? sd.wait() alternative below if needed.
                    # Alternative: Use sd.wait() if blocking is desired *after* checking interrupt once?
                    # sd.wait() # This blocks until playback is complete. Combine with interrupt check?

                    logger.debug("Speech Worker: Finished utterance playback.")

                except sd.PortAudioError as pae:
                     logger.error(f"Speech Worker: PortAudioError during playback: {pae}")
                     # Potentially try to recover or disable further playback?
                except Exception as e:
                    logger.exception("Speech Worker: Error during audio playback.")

                finally:
                     # Ensure task is marked done regardless of playback success/interrupt
                     self.speak_queue.task_done()


            except Exception as e:
                logger.exception("Unexpected error in speech worker thread.")
                # Avoid crashing the thread if possible
                if not self.stop_event.is_set(): time.sleep(1)

        logger.info("Speech worker thread finished.")


    def wait_for_speech_to_finish(self):
        """Blocks until the speech queue is empty and current playback finishes."""
        if not self.tts_pipeline or not SOUNDDEVICE_AVAILABLE: return
        logger.debug("Waiting for speech queue to empty...")
        self.speak_queue.join() # Wait for all items to be processed by worker
        # Additionally, wait for the *last* utterance's playback to finish
        if SOUNDDEVICE_AVAILABLE and sd.get_stream().active:
             logger.debug("Waiting for current playback to finish...")
             # Cannot simply sd.wait() here as it blocks this thread, worker might need to process more.
             # Polling is one way:
             while sd.get_stream().active:
                 time.sleep(0.1)
             logger.debug("Current playback finished.")
        logger.debug("Speech queue empty and playback finished.")

    def shutdown(self):
        logger.info("Shutting down Speech Manager.")
        self.stop_event.set()
        self.interrupt_event.set() # Signal interrupt to stop any current playback
        # Put sentinel value to ensure worker wakes up if blocked on queue.get()
        self.speak_queue.put(None)
        if self.speak_thread.is_alive():
             logger.debug("Joining speech worker thread...")
             self.speak_thread.join(timeout=3.0) # Increased timeout
             if self.speak_thread.is_alive():
                 logger.warning("Speech worker thread did not exit cleanly.")
        # Release sounddevice resources? Usually not needed explicitly unless specific handles used.
        logger.info("Speech Manager shutdown complete.")


# --- Assistant Class (Refactored, using updated components) ---
class Assistant:
    def __init__(self, config_path="config.yaml"):
        logger.info("Initializing Assistant...")
        try:
             self.config = self._load_config(config_path)
             self._configure_libraries()
        except Exception as e:
             logger.critical(f"Failed to load configuration or configure libraries: {e}", exc_info=True)
             raise RuntimeError("Configuration failed.") from e # Re-raise critical error

        # --- Core Components ---
        # Init Map Manager first, as other components might depend on it
        self.node_map_manager = self._init_map_manager() # Can raise FileNotFoundError

        # Init Speech Manager (handles TTS loading later)
        self.speech_manager = SpeechManager(self.config)

        # Init Navigation Manager (Requires map and speech)
        if self.node_map_manager:
             self.navigation_manager = NavigationManager(
                 self.speech_manager.speak, # Pass the speak function
                 self.node_map_manager,
                 self.config
            )
        else:
             logger.warning("NodeMapManager failed to initialize. Navigation features will be disabled.")
             self.navigation_manager = None

        # --- LLM & Tools ---
        # Init LLMs (Tool Agent and potentially Vision)
        self.agent_llm = self._init_llm('tool_llm_model_id', 'Tool-Using LLM')
        self.vision_llm = self._init_llm('vision_llm_model_id', 'Vision LLM', required=False) # Vision is optional

        # Define tools only if components they rely on are available
        self.tools = self._define_tools()
        if self.agent_llm and self.tools:
             self.llm_with_tools = self.agent_llm.bind_tools(self.tools)
        else:
             self.llm_with_tools = self.agent_llm # Fallback to LLM without tools if setup failed
             if not self.agent_llm: logger.error("Agent LLM not initialized. Tool binding skipped.")
             if not self.tools: logger.error("No tools defined or available. Tool binding skipped.")


        self.chat_history = [SystemMessage(content=self.config.get('system_prompt', "You are a helpful assistant for visually impaired users providing scene descriptions, OCR, and step-by-step indoor navigation. Use tools proactively when asked to 'describe', 'read', 'navigate', or 'where am i'. Respond concisely."))]

        # Ensure TTS is loaded *before* starting the main loop if possible
        # Moved to start of run() to allow rest of init to succeed first
        # try:
        #      self.speech_manager.load_tts()
        # except RuntimeError as e:
        #      logger.error(f"TTS failed to load during init: {e}. Speech will be unavailable.")

        logger.info("Assistant Initialized.")

    def _load_config(self, config_path):
        logger.info(f"Loading configuration from {config_path}")
        if not os.path.exists(config_path):
            raise FileNotFoundError(f"Configuration file not found: {config_path}")
        try:
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            # Set environment variables from config if needed
            if 'openai_api_key' in config:
                 os.environ["OPENAI_API_KEY"] = config['openai_api_key']
                 logger.info("Set OPENAI_API_KEY from config.")
            # Add other env vars if necessary (e.g., specific TTS API keys)
            return config
        except yaml.YAMLError as e:
            logger.exception(f"Error parsing YAML configuration file: {config_path}")
            raise RuntimeError("Invalid configuration format.") from e
        except Exception as e:
            logger.exception(f"Error loading configuration file: {config_path}")
            raise RuntimeError("Failed to load configuration.") from e

    def _configure_libraries(self):
        # Configure Tesseract path
        tesseract_path = self.config.get('tesseract_cmd_path')
        if tesseract_path:
            if os.path.exists(tesseract_path):
                 try:
                    pytesseract.pytesseract.tesseract_cmd = tesseract_path
                    logger.info(f"Tesseract path set to: {tesseract_path}")
                 except Exception as e:
                    logger.error(f"Failed to set Tesseract path '{tesseract_path}': {e}")
            else:
                logger.warning(f"Tesseract path specified in config but not found: {tesseract_path}. OCR may fail.")
        else:
            logger.info("Tesseract path not specified in config, using system default.")

        # Configure sounddevice default device (optional)
        # Can set sd.default.device here if needed, based on config

    def _init_map_manager(self):
        try:
            map_path = self.config.get('map_filepath', 'map.json') # Default filename
            return NodeMapManager(map_path) # Can raise FileNotFoundError
        except FileNotFoundError as e:
            logger.error(f"Map file error: {e}. Navigation disabled.", exc_info=True)
            raise # Propagate error to prevent running without a map
        except Exception as e:
            logger.error(f"Failed to initialize NodeMapManager: {e}. Navigation disabled.", exc_info=True)
            # Depending on severity, could return None or raise
            raise RuntimeError("Failed to initialize map manager.") from e


    def _init_llm(self, model_config_key: str, llm_name: str, required: bool = True) -> Optional[ChatOpenAI]:
        """Initializes a ChatOpenAI LLM instance based on config."""
        model_id = self.config.get(model_config_key)
        base_url = self.config.get('lm_studio_base_url') # Assuming one base URL for potentially multiple models
        api_key = self.config.get('openai_api_key', 'lm-studio') # Use configured key or default

        if not model_id:
            if required:
                logger.error(f"{llm_name} model ID ('{model_config_key}') not found in config. Cannot initialize.")
                raise ValueError(f"Missing required LLM config: {model_config_key}")
            else:
                logger.warning(f"{llm_name} model ID ('{model_config_key}') not found in config. {llm_name} features disabled.")
                return None
        if not base_url and required: # Base URL might be needed depending on setup
             logger.error(f"LM Studio base URL ('lm_studio_base_url') not found in config. Cannot initialize {llm_name}.")
             raise ValueError("Missing required LLM config: lm_studio_base_url")

        logger.info(f"Initializing {llm_name} (Model: {model_id}, BaseURL: {base_url})...")
        try:
            llm = ChatOpenAI(
                base_url=base_url,
                api_key=api_key,
                model=model_id,
                temperature=self.config.get('llm_temperature', 0.2), # Configurable temp
                max_tokens=self.config.get('llm_max_tokens', 500), # Configurable max tokens
                # streaming=False, # Keep streaming off for tool use compatibility unless handled carefully
                # timeout=self.config.get('llm_timeout', 60) # Configurable timeout
            )
            # Perform a simple test call? (Optional, adds startup time)
            # llm.invoke("Hello")
            logger.info(f"{llm_name} initialized successfully.")
            return llm
        except Exception as e:
            logger.exception(f"Failed to initialize {llm_name}.")
            if required:
                raise RuntimeError(f"{llm_name} init failed.") from e
            else:
                 return None # Allow graceful degradation if not required

    def _define_tools(self):
        """Defines LangChain tools based on available components."""
        available_tools = []

        # Scene Description Tool
        if self.vision_llm: # Only add if vision LLM is available
            @tool
            def scene_description_tool() -> str:
                """Describes the current scene using the camera. Provides details about layout, objects, potential obstacles, and pathways relevant to a visually impaired user."""
                # Navigation pause/resume is handled by user commands ('pause', 'resume')
                # Direct commands are faster than interrupting the LLM flow for this.
                return self._execute_scene_description()
            available_tools.append(scene_description_tool)
        else:
            logger.warning("Scene Description tool disabled: Vision LLM not configured.")

        # OCR Tool
        # Check if tesseract is likely available (basic check)
        try:
            pytesseract.get_tesseract_version()
            tesseract_ok = True
        except pytesseract.TesseractNotFoundError:
            logger.warning("OCR tool disabled: Tesseract not found or configured correctly.")
            tesseract_ok = False
        except Exception as e:
            logger.warning(f"OCR tool potentially disabled: Error checking Tesseract ({e}).")
            tesseract_ok = False # Assume unavailable on error

        if tesseract_ok:
            @tool
            def ocr_text_tool() -> str:
                """Reads text detected in front of the user using the camera. Useful for signs, labels, or documents."""
                return self._execute_ocr()
            available_tools.append(ocr_text_tool)
        # else: Tool not added if Tesseract seems unavailable

        # Navigation Tool
        if self.navigation_manager: # Only add if nav manager is available
            @tool
            def indoor_navigation_tool(destination: str) -> str:
                """Starts indoor navigation to a specific named location within the mapped area (e.g., 'reception', 'elevator', 'room 101', 'exit'). Specify the exact name from the map if known."""
                return self._execute_indoor_navigation_request(destination)
            available_tools.append(indoor_navigation_tool)
        else:
            logger.warning("Indoor Navigation tool disabled: Navigation Manager not available.")

        # Where Am I Tool
        if self.node_map_manager: # Requires map to give location context
            @tool
            def where_am_i_tool() -> str:
                """Reports the user's current estimated location based on the map or current navigation status if navigating."""
                return self._execute_where_am_i()
            available_tools.append(where_am_i_tool)
        else:
             logger.warning("Where Am I tool disabled: Node Map Manager not available.")


        if not available_tools:
            logger.error("No tools could be defined due to missing dependencies or configuration.")
            return [] # Return empty list

        logger.info(f"Defined tools: {[t.name for t in available_tools]}")
        return available_tools

    # --- Tool Execution Logic ---
    def _execute_scene_description(self) -> str:
        """Captures image, calls vision LLM, returns description text."""
        if not self.vision_llm: return "Scene description feature is currently unavailable."
        try:
            image_path = capture_image(self.config.get('image_capture_path', 'capture_scene.jpg'))
            if not image_path:
                self.speech_manager.speak("Failed to capture image for description.", interrupt=True)
                return "Failed to capture image."

            description = sceene_description_with_tts_modified(self.vision_llm, image_path)

            # Speak the description (non-interrupting unless specified)
            self.speech_manager.speak(description, interrupt=False)

            # Clean up captured image? (Optional)
            # if os.path.exists(image_path): os.remove(image_path)

            # Return text for LLM context
            return description
        except Exception as e:
            error_msg = f"Error describing scene: {e}"
            logger.exception(error_msg)
            self.speech_manager.speak("Sorry, I encountered an error while describing the scene.", interrupt=True)
            return error_msg # Return error detail to LLM

    def _execute_ocr(self) -> str:
        """Captures image, runs OCR, speaks result, returns text."""
        try:
            image_path = capture_image(self.config.get('image_capture_path', 'capture_ocr.jpg'))
            if not image_path or not os.path.exists(image_path):
                 self.speech_manager.speak("Failed to capture image for reading.", interrupt=True)
                 raise FileNotFoundError("Failed image capture for OCR.")

            # Use pytesseract
            text = pytesseract.image_to_string(Image.open(image_path))
            result_text = text.strip()

            if not result_text:
                response = "No readable text found."
                self.speech_manager.speak(response, interrupt=True)
            else:
                # Speak the found text - use interrupt as OCR is usually a specific request.
                logger.info(f"OCR Result: '{result_text[:100]}...'")
                self.speech_manager.speak(f"Detected text: {result_text}", interrupt=True)

             # Clean up captured image? (Optional)
             # if os.path.exists(image_path): os.remove(image_path)

            # Return text for LLM context
            return result_text if result_text else response # Return "No text found" if empty
        except pytesseract.TesseractNotFoundError:
             error_msg = "Error reading text: Tesseract is not installed or configured correctly."
             logger.error(error_msg)
             self.speech_manager.speak(error_msg, interrupt=True)
             return error_msg
        except Exception as e:
             error_msg = f"Error reading text: {e}"
             logger.exception(error_msg)
             self.speech_manager.speak("Sorry, I encountered an error while reading text.", interrupt=True)
             return error_msg

    def _execute_indoor_navigation_request(self, destination: str) -> str:
        """Handles the request to start navigation via the tool."""
        if not self.navigation_manager or not self.node_map_manager:
            return "Navigation system is currently unavailable."
        if self.navigation_manager.is_navigating:
            # Maybe ask if they want to cancel the current one? For now, reject.
            msg = f"You are already navigating towards {self.navigation_manager.navigation_destination_name}. Please cancel the current navigation first if you want to go to {destination}."
            self.speech_manager.speak(msg)
            return msg

        # 1. Get Current Location (Simulation)
        current_node_id = get_current_location_node_id()
        if not current_node_id:
            msg = "Cannot start navigation: Unable to determine your current location."
            self.speech_manager.speak(msg, interrupt=True)
            return msg

        # 2. Find Destination Node ID
        # Try direct name lookup first
        destination_node_id = self.node_map_manager.get_node_id_by_name(destination)
        if not destination_node_id:
             # Basic fuzzy matching fallback (optional, can be complex)
             # Simplification: LLM might re-invoke if it gets an ambiguous name.
             msg = f"Sorry, I could not find a location named '{destination}' on the map. Please try a different name or check available locations."
             self.speech_manager.speak(msg, interrupt=True)
             # Consider listing known locations here if map is small?
             # known_names = list(self.node_map_manager._node_name_to_id.keys())
             # self.speech_manager.speak(f"Known locations include: {', '.join(known_names[:5])}...", interrupt=False)
             return msg

        # 3. Check if already at destination
        if current_node_id == destination_node_id:
            node_data = self.node_map_manager.get_node_data(destination_node_id)
            dest_name = node_data.get('name', destination) if node_data else destination
            msg = f"You are already at {dest_name}."
            self.speech_manager.speak(msg, interrupt=False)
            return msg

        # 4. Get Current Heading (Simulation)
        current_head = get_current_heading()

        # 5. Calculate Path
        logger.info(f"Calculating path from {current_node_id} to {destination_node_id} ({destination})")
        path = self.node_map_manager.get_shortest_path(current_node_id, destination_node_id)
        if not path:
            msg = f"Sorry, I could not find a navigable path from your current location to {destination}."
            self.speech_manager.speak(msg, interrupt=True)
            return msg

        # 6. Generate Instructions
        instructions = generate_navigation_instructions(path, self.node_map_manager, current_head)
        if not instructions:
            msg = f"Found a path to {destination}, but failed to generate navigation instructions."
            logger.error(f"Instruction generation failed for path: {path}")
            self.speech_manager.speak(msg, interrupt=True)
            return msg

        # 7. Start Navigation Session using NavigationManager
        # The NavigationManager will handle speaking the first instruction.
        destination_data = self.node_map_manager.get_node_data(destination_node_id)
        actual_dest_name = destination_data.get('name', destination) if destination_data else destination

        if self.navigation_manager.start_session(path, instructions, actual_dest_name, destination_node_id):
            # Return a confirmation message for the LLM context. Speech is handled by start_session.
            return f"Navigation started towards {actual_dest_name}. The first instruction has been spoken."
        else:
             # start_session failed (logged internally)
             msg = f"Sorry, I encountered an internal error and could not start navigation to {actual_dest_name}."
             self.speech_manager.speak(msg, interrupt=True)
             return msg

    def _execute_where_am_i(self) -> str:
         """Handles the 'Where am I?' request based on navigation status or nearest node."""
         status = "Unable to determine location status." # Default

         # If navigating, get status from NavigationManager
         if self.navigation_manager and self.navigation_manager.is_navigating:
              status = self.navigation_manager.get_status()
              logger.info(f"Where am I? Reporting navigation status: {status}")
         # If not navigating, report based on nearest node
         elif self.node_map_manager:
              current_node_id = get_current_location_node_id() # Simulation
              if current_node_id:
                   node_data = self.node_map_manager.get_node_data(current_node_id)
                   if node_data:
                       loc_name = node_data.get('name')
                       loc_type = node_data.get('type')
                       landmark = node_data.get('landmark')
                       coord_info = f"(approx {node_data.get('x', '?'):.1f}, {node_data.get('y','?'):.1f})" if 'x' in node_data else ""

                       if loc_name:
                            status = f"You are currently near {loc_name} {coord_info}"
                       else:
                            status = f"You are currently near node {current_node_id} {coord_info}"

                       if loc_type and loc_type not in ['Intersection', 'Area', '']:
                           status += f", which is a {loc_type}."
                       elif landmark:
                           status += f", close to the landmark: {landmark}."
                       else:
                            status += "." # Append period if no extra info.

                       # Add heading info?
                       current_head = get_current_heading()
                       status += f" You are facing approximately {self._get_heading_direction(current_head)} ({current_head:.0f} degrees)."

                       logger.info(f"Where am I? Reporting location: {status}")
                   else:
                        status = f"I know you are near node {current_node_id}, but I don't have specific details for it."
                        logger.warning(f"Where am I? Node data not found for simulated node {current_node_id}")
              else:
                   status = "I'm unable to determine your current location on the map at the moment."
                   logger.warning("Where am I? Failed to get current location node ID from simulation.")
         else:
              status = "Location services are unavailable as the map could not be loaded."

         # Speak the determined status (don't interrupt ongoing navigation instructions unless needed)
         self.speech_manager.speak(status, interrupt=False)
         return status # Return text for LLM context

    def _get_heading_direction(self, heading: float) -> str:
         """Converts heading degree to approximate cardinal direction."""
         heading = heading % 360
         if 337.5 <= heading or heading < 22.5: return "East"
         if 22.5 <= heading < 67.5: return "Northeast"
         if 67.5 <= heading < 112.5: return "North"
         if 112.5 <= heading < 157.5: return "Northwest"
         if 157.5 <= heading < 202.5: return "West"
         if 202.5 <= heading < 247.5: return "Southwest"
         if 247.5 <= heading < 292.5: return "South"
         if 292.5 <= heading < 337.5: return "Southeast"
         return "an unknown direction"


    def _handle_command(self, command: str) -> bool:
        """Processes a user command, handling direct actions or invoking the LLM agent. Returns False to quit."""
        if not command: return True # Ignore empty commands
        command_lower = command.strip().lower()
        logger.info(f"Received command: '{command}'")

        # --- Direct Command Handling (Bypass LLM for speed/reliability) ---
        if command_lower == 'quit':
            self.speech_manager.speak("Exiting.")
            return False # Signal to exit run loop

        # Navigation Lifecycle Commands
        if self.navigation_manager:
            if command_lower == 'cancel' or command_lower == 'cancel navigation' or command_lower == 'stop navigation':
                if self.navigation_manager.cancel_session(): pass # Already speaks
                else: self.speech_manager.speak("There is no active navigation to cancel.")
                return True # Command handled
            elif command_lower == 'pause' or command_lower == 'pause navigation':
                 if self.navigation_manager.pause_session(): pass # Already speaks
                 else: self.speech_manager.speak("Cannot pause. Are you navigating?") # More informative
                 return True
            elif command_lower == 'resume' or command_lower == 'resume navigation':
                 if self.navigation_manager.resume_session(): pass # Already speaks
                 else: self.speech_manager.speak("Cannot resume. Was navigation paused?")
                 return True
            elif command_lower == 'repeat' or command_lower == 'repeat instruction':
                 if self.navigation_manager.repeat_instruction(): pass # Already speaks
                 else: self.speech_manager.speak("Nothing to repeat. Are you navigating?")
                 return True
            # Where am I can also be handled directly for speed
            elif command_lower == 'where am i' or command_lower == 'what is my location' or command_lower == 'location':
                 self._execute_where_am_i() # Handles speech
                 return True # Command handled

        # --- LLM Agent Interaction (for other commands or if direct failed) ---
        if not self.llm_with_tools or not self.agent_llm:
            self.speech_manager.speak("My language understanding capabilities are offline.", interrupt=True)
            logger.error("Cannot process command with LLM: Agent LLM or tools not available.")
            return True # Continue running, but can't process complex commands

        try:
            # Add user message to history
            self.chat_history.append(HumanMessage(content=command))

            logger.info("Invoking LLM agent...")
            # Use the bound LLM (which handles tool selection)
            ai_response: AIMessage = self.llm_with_tools.invoke(self.chat_history)
            self.chat_history.append(ai_response)

            # Check if the AI response contains tool calls
            if ai_response.tool_calls and self.tools:
                logger.info(f"Agent initiated tool calls: {ai_response.tool_calls}")
                tool_outputs = []
                # Execute tools sequentially (can be parallelized if tools are independent and thread-safe)
                for tool_call in ai_response.tool_calls:
                    tool_name = tool_call['name']
                    tool_args = tool_call['args']
                    tool_id = tool_call['id'] # Important for response mapping

                    # Find the corresponding tool object
                    selected_tool = next((t for t in self.tools if t.name == tool_name), None)

                    if selected_tool:
                        tool_result_content = ""
                        try:
                            # Invoke the tool's underlying function (_execute_*)
                            # Tool methods should handle their own primary speech output.
                            # The return value is primarily for the LLM's context.
                            logger.info(f"Executing tool: {tool_name} with args: {tool_args}")
                            tool_result = selected_tool.invoke(tool_args) # Calls the decorated function
                            tool_result_content = str(tool_result)
                            logger.info(f"Tool {tool_name} executed. Result preview: '{tool_result_content[:100]}...'")
                        except Exception as e:
                            logger.exception(f"Error executing tool {tool_name}")
                            tool_result_content = f"Error executing tool {tool_name}: {e}"
                            # Speak the error? Maybe let LLM summarize errors?
                            # self.speech_manager.speak(f"Sorry, I encountered an error with the {tool_name} tool.", interrupt=True)

                        # Append result message for the LLM
                        tool_outputs.append(ToolMessage(content=tool_result_content, tool_call_id=tool_id))
                    else:
                        logger.error(f"Agent tried to call unknown tool: {tool_name}")
                        tool_outputs.append(ToolMessage(content=f"Error: Tool '{tool_name}' not found.", tool_call_id=tool_id))

                # If any tools were called, send results back to LLM for final response
                if tool_outputs:
                    self.chat_history.extend(tool_outputs)
                    logger.info("Invoking agent again with tool results...")
                    final_ai_response = self.llm_with_tools.invoke(self.chat_history)
                    self.chat_history.append(final_ai_response)
                    final_response_to_speak = final_ai_response.content if hasattr(final_ai_response, 'content') else None
                else: # Should not happen if tool_calls existed but none were valid
                    final_response_to_speak = "I tried to use a tool, but something went wrong."

            else: # No tool call in the initial response
                final_response_to_speak = ai_response.content if hasattr(ai_response, 'content') else None

            # Speak the final LLM text response (if any)
            if final_response_to_speak:
                 logger.info(f"Agent final response: '{final_response_to_speak[:100]}...'")
                 # Use interrupt=False unless it's critical information following a tool error?
                 self.speech_manager.speak(final_response_to_speak, interrupt=False)
            else:
                 logger.info("Agent did not provide a final text response (might have just invoked tools).")


        except Exception as e:
            logger.exception("An error occurred during LLM agent interaction.")
            self.speech_manager.speak("Sorry, I encountered an issue processing your request.", interrupt=True)
            # Add error message to history?
            self.chat_history.append(AIMessage(content=f"[Error processing last command: {e}]"))
        finally:
             # Trim history to prevent excessive length
             self._trim_chat_history()

        return True # Continue running

    def _trim_chat_history(self):
        """Keeps chat history within a configured size limit."""
        max_hist = self.config.get('max_chat_history', 10) # Keep 1 System msg + N recent turns
        # Ensure max_hist is at least 2 (System + 1 turn)
        max_hist = max(max_hist, 2)
        if len(self.chat_history) > max_hist:
             # Keep the first message (System) and the last N-1 messages
             logger.debug(f"Trimming chat history from {len(self.chat_history)} to {max_hist} messages.")
             self.chat_history = [self.chat_history[0]] + self.chat_history[-(max_hist-1):]


    def run(self):
        """Main interaction loop."""
        # --- Ensure TTS is loaded before starting ---
        try:
            self.speech_manager.load_tts()
            # Only speak ready if TTS is confirmed available
            if self.speech_manager.tts_pipeline and SOUNDDEVICE_AVAILABLE:
                self.speech_manager.speak("Assistant ready.", interrupt=True)
            else:
                 logger.warning("Assistant starting without speech capabilities.")
                 print("Assistant ready (WARNING: Text-to-speech is unavailable).") # Console fallback
        except RuntimeError as e:
            # If load_tts raises, it's critical (e.g., sounddevice missing)
            logger.critical(f"Failed to load critical TTS/Audio component: {e}. Cannot run assistant.", exc_info=True)
            print(f"ERROR: Failed to initialize audio: {e}. Assistant cannot run.")
            return # Exit run method

        # --- Start command input thread ---
        input_thread = threading.Thread(target=command_input_thread, daemon=True)
        input_thread.start()

        last_command_check_time = time.time()
        keep_running = True

        # --- Main Loop ---
        while keep_running:
            try:
                current_time = time.time()

                # --- 1. Navigation Update Logic (if navigating) ---
                if self.navigation_manager and self.navigation_manager.is_navigating and not self.navigation_manager.is_paused:
                    # Update simulation state (passing map manager for heading calc)
                    simulate_movement(self.navigation_manager.navigation_path, self.node_map_manager)

                    # Check progress and handle rerouting requests
                    reroute_request = self.navigation_manager.update_progress()
                    if reroute_request == "reroute":
                        logger.info("Reroute requested by NavigationManager.")
                        # LLM/Tool should handle speaking the "off route" message
                        # Get current location *after* simulation update might have moved us
                        current_node_id = get_current_location_node_id()
                        if not current_node_id:
                             logger.error("Reroute failed: Cannot get current location.")
                             self.speech_manager.speak("I need to reroute, but I cannot determine your current location.", interrupt=True)
                        elif not self.navigation_manager.navigation_destination_name:
                            logger.error("Reroute failed: Destination name is missing.")
                            self.speech_manager.speak("I need to reroute, but I seem to have forgotten the destination.", interrupt=True)
                        else:
                            # Speak intent *before* potentially long calculation
                            self.speech_manager.speak("It seems you are off route. Recalculating...", interrupt=True)
                            # Cancel current session silently before starting new one
                            self.navigation_manager.cancel_session(speak_cancel=False)
                            # Use the internal execution method directly - simulates calling the tool
                            # Use the *name* of the original destination for the request
                            logger.info(f"Calling internal navigation request for reroute to: {self.navigation_manager.navigation_destination_name}")
                            reroute_result = self._execute_indoor_navigation_request(self.navigation_manager.navigation_destination_name)
                            logger.info(f"Reroute execution result (for LLM context): {reroute_result}")
                            # Speech for the *new* route is handled within _execute_indoor_navigation_request

                # --- 2. Command Processing Logic ---
                # Check for user commands periodically
                command_check_interval = self.config.get('command_check_interval', 0.2) # Check more frequently?
                if current_time - last_command_check_time > command_check_interval:
                     command = listen_for_command_non_blocking()
                     last_command_check_time = current_time
                     if command:
                         # Pass command to handler, which returns False to quit
                         keep_running = self._handle_command(command)
                         if not keep_running:
                             break # Exit loop immediately on quit command


                # --- 3. Prevent Busy-Waiting ---
                # Sleep for a short duration to yield CPU
                # Adjust sleep time based on required responsiveness vs CPU usage
                # Must be shorter than command_check_interval and nav_update_interval
                time.sleep(0.1)

            except KeyboardInterrupt:
                logger.info("Keyboard interrupt received. Shutting down.")
                keep_running = False
            except Exception as e:
                logger.exception("Unhandled exception in main loop. Attempting to continue.")
                self.speech_manager.speak("An unexpected error occurred. Please try again.", interrupt=True)
                # Avoid continuous rapid errors
                time.sleep(1)


        logger.info("Main run loop finished.")


    def shutdown(self):
        """Cleanly shutdown resources."""
        logger.info("Shutting down Assistant...")
        # Ensure speech finishes and thread closes
        if hasattr(self, 'speech_manager') and self.speech_manager:
             logger.info("Waiting for speech to finish...")
             self.speech_manager.wait_for_speech_to_finish()
             logger.info("Shutting down speech manager...")
             self.speech_manager.shutdown()
        else:
            logger.info("Speech manager not available or already shut down.")

        # Add any other cleanup needed (e.g., close sensor connections, release camera)
        # cv2.destroyAllWindows() # If using OpenCV windows

        # Cancel navigation if active
        if self.navigation_manager and self.navigation_manager.is_navigating:
            logger.info("Cancelling active navigation session during shutdown.")
            self.navigation_manager.cancel_session(speak_cancel=False) # Don't speak during shutdown

        logger.info("Assistant shutdown complete.")


# --- Main Execution ---
if __name__ == "__main__":
    assistant = None
    try:
        # Create necessary files if they don't exist (optional, for ease of first run)
        if not os.path.exists("config.yaml"):
            print("config.yaml not found. Creating a default example.")
            default_config = """
# --- API Keys & Endpoints ---
# openai_api_key: "sk-..." # Uncomment and add your OpenAI key if using OpenAI models
lm_studio_base_url: "http://localhost:1234/v1" # Default for LM Studio

# --- Model IDs (Match models loaded in LM Studio or OpenAI) ---
tool_llm_model_id: "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF" # Example tool-using model
vision_llm_model_id: "microsoft/Phi-3-vision-128k-instruct" # Example vision model (ensure loaded)

# --- TTS Configuration ---
tts_device: "cpu" # Device for TTS model ('cpu' or 'cuda')
tts_voice: "onyx" # Voice for TTS (specific to model, e.g., OpenAI TTS voices: alloy, echo, fable, onyx, nova, shimmer)
audio_sample_rate: 24000 # Sample rate expected by TTS/playback

# --- File Paths ---
map_filepath: "map.json" # Path to the indoor map file
tesseract_cmd_path: "" # Optional: Full path to tesseract executable if not in PATH (e.g., "C:/Program Files/Tesseract-OCR/tesseract.exe")
image_capture_path: "captured_image.jpg" # Default path to save captured images

# --- Behaviour Configuration ---
system_prompt: "You are a helpful indoor assistant for visually impaired users. Use tools proactively for 'describe', 'read', 'nav', 'where am i'. Be concise."
max_chat_history: 10 # Max number of messages (turns) to keep in history (including System)
navigation_update_interval: 2.0 # Seconds between checking navigation progress/location
off_route_threshold_nodes: 2 # Number of consecutive off-route location checks before triggering reroute
# command_check_interval: 0.2 # Seconds between checking for user input
# llm_temperature: 0.2
# llm_max_tokens: 500
"""
            with open("config.yaml", "w") as f: f.write(default_config)
        if not os.path.exists("map.json"):
             print("map.json not found. Creating a minimal example map.")
             # Example Map: Node 1 (Entrance) <-> Node 2 (Intersection) <-> Node 3 (Elevator)
             default_map = {
                 "nodes": [
                     {"id": "node1", "name": "Entrance", "type": "Entrance", "x": 0, "y": 0},
                     {"id": "node2", "name": "Hallway Intersection", "type": "Intersection", "x": 5, "y": 0},
                     {"id": "node3", "name": "Elevator Lobby", "type": "Area", "x": 10, "y": 0, "landmark": "the elevator buttons"},
                     {"id": "node4", "name": "Reception Desk", "type": "Reception", "x": 5, "y": 5, "landmark": "the front desk"}
                 ],
                 "links": [ # Define connections (implicitly bidirectional here due to graph logic)
                     {"startNode": "node1", "endNode": "node2"},
                     {"startNode": "node2", "endNode": "node3"},
                     {"startNode": "node2", "endNode": "node4"}
                 ]
             }
             with open("map.json", "w") as f: json.dump(default_map, f, indent=2)

        # --- Initialize and Run Assistant ---
        assistant = Assistant(config_path="config.yaml")
        assistant.run()

    except (FileNotFoundError, ValueError, RuntimeError) as e:
        # Catch initialization errors specifically
        logger.critical(f"Assistant initialization failed: {e}", exc_info=False) # Less verbose log for common init issues
        print(f"\nERROR: Could not start the assistant. Please check configuration and file paths.")
        print(f"Details: {e}")
    except Exception as e:
        # Catch unexpected errors during runtime or init
        logger.critical(f"An unexpected critical error occurred: {e}", exc_info=True)
        print(f"\nFATAL ERROR: {e}")
    finally:
        # --- Ensure Shutdown is Called ---
        if assistant:
            logger.info("Attempting final shutdown.")
            assistant.shutdown()
        else:
             logger.info("Assistant object was not successfully created.")
        logger.info("Application exiting.")
        # Add a small delay to ensure logs are flushed
        time.sleep(0.5)

2025-04-11 10:13:22,724 - INFO - __main__ - Initializing Assistant...
2025-04-11 10:13:22,725 - INFO - __main__ - Loading configuration from config.yaml
2025-04-11 10:13:22,738 - INFO - __main__ - Set OPENAI_API_KEY from config.
2025-04-11 10:13:22,739 - INFO - __main__ - Tesseract path not specified in config, using system default.
2025-04-11 10:13:22,739 - INFO - __main__ - Initializing NodeMapManager with map: map.json
2025-04-11 10:13:22,750 - ERROR - __main__ - Failed to initialize NodeMapManager: 'NodeMapManager' object has no attribute '_node_id_to_data'. Navigation disabled.
Traceback (most recent call last):
  File "C:\Users\rohit\AppData\Local\Temp\ipykernel_22260\2963068995.py", line 1034, in _init_map_manager
    return NodeMapManager(map_path) # Can raise FileNotFoundError
  File "C:\Users\rohit\AppData\Local\Temp\ipykernel_22260\2963068995.py", line 145, in __init__
    self._build_graph()
  File "C:\Users\rohit\AppData\Local\Temp\ipykernel_22260\2963068995.py", line 230,

map.json not found. Creating a minimal example map.

ERROR: Could not start the assistant. Please check configuration and file paths.
Details: Failed to initialize map manager.


In [2]:
import cv2
import json
import math
import os
import numpy as np
import argparse

# --- Configuration ---
NODE_COLOR = (0, 255, 0)      # Green for nodes
NODE_RADIUS = 8
TEXT_COLOR = (255, 255, 255) # White text
LINK_COLOR = (255, 0, 0)      # Blue for links
HIGHLIGHT_COLOR = (0, 255, 255) # Yellow for selection
MAX_CLICK_DISTANCE_NODE = 20 # Max distance in pixels to associate a click with a node

# --- Global State Variables ---
image = None             # Original loaded image
display_image = None     # Image copy to draw on
nodes = []               # List to store node dictionaries
links = []               # List to store link dictionaries (pairs of node IDs)
node_counter = 1         # Simple counter for unique node IDs

mode = 'NODE'            # Current interaction mode: 'NODE', 'LINK_START', 'LINK_END', 'SCALE_START', 'SCALE_END'
scale_points = []        # Store the two points clicked for scaling
scale_pixels_per_meter = None # Calculated scale factor (pixels / meter)

link_start_node = None   # Temporarily store the first node clicked for linking

# --- Helper Functions ---

def calculate_distance(p1, p2):
    """Calculates Euclidean distance between two points (pixels)."""
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def get_real_distance(node1, node2):
    """Calculates real-world distance between two nodes using the scale."""
    if not node1 or not node2:
        return float('inf')
    pixel_dist = calculate_distance((node1['pixel_x'], node1['pixel_y']), (node2['pixel_x'], node2['pixel_y']))
    if scale_pixels_per_meter and scale_pixels_per_meter > 0:
        return pixel_dist / scale_pixels_per_meter
    else:
        # Return pixel distance if scale is not set (less meaningful for navigation)
        print("WARN: Scale not set. Using pixel distance for link weight.")
        return pixel_dist

def find_nearest_node(x, y, max_dist=MAX_CLICK_DISTANCE_NODE):
    """Finds the nearest node within max_dist pixels of the click."""
    nearest_node = None
    min_dist = float('inf')
    for node in nodes:
        dist = calculate_distance((x, y), (node['pixel_x'], node['pixel_y']))
        if dist < min_dist and dist <= max_dist:
            min_dist = dist
            nearest_node = node
    return nearest_node

def get_node_by_id(node_id):
    """Retrieves a node dictionary by its ID."""
    for node in nodes:
        if node['id'] == node_id:
            return node
    return None

def redraw_display():
    """Redraws the image with current nodes and links."""
    global display_image
    display_image = image.copy() # Start fresh from original

    # Draw Links
    for link in links:
        node1 = get_node_by_id(link['startNode'])
        node2 = get_node_by_id(link['endNode'])
        if node1 and node2:
            pt1 = (node1['pixel_x'], node1['pixel_y'])
            pt2 = (node2['pixel_x'], node2['pixel_y'])
            cv2.line(display_image, pt1, pt2, LINK_COLOR, 2)

    # Draw Nodes (and highlight selected start node for linking)
    for node in nodes:
        center = (node['pixel_x'], node['pixel_y'])
        color = NODE_COLOR
        radius = NODE_RADIUS
        # Highlight if it's the starting node for a link
        if mode == 'LINK_END' and link_start_node and node['id'] == link_start_node['id']:
             color = HIGHLIGHT_COLOR
             radius = NODE_RADIUS + 3 # Make it slightly bigger

        cv2.circle(display_image, center, radius, color, -1) # Filled circle
        # Draw node ID text nearby
        text_pos = (center[0] + NODE_RADIUS + 2, center[1] + NODE_RADIUS + 2)
        cv2.putText(display_image, node['id'], text_pos, cv2.FONT_HERSHEY_SIMPLEX, 0.5, TEXT_COLOR, 1, cv2.LINE_AA)

    # Draw scale points if in scaling mode
    if mode in ['SCALE_START', 'SCALE_END'] and scale_points:
        for pt in scale_points:
            cv2.drawMarker(display_image, pt, HIGHLIGHT_COLOR, cv2.MARKER_CROSS, 15, 2)

    cv2.imshow("Floor Plan Map Generator", display_image)


def mouse_callback(event, x, y, flags, param):
    """Handles mouse click events."""
    global mode, node_counter, link_start_node, scale_points, scale_pixels_per_meter

    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Clicked at ({x}, {y}) in mode: {mode}")

        # === NODE MODE: Add a new node ===
        if mode == 'NODE':
            node_id = f"node{node_counter}"
            print(f"Adding node {node_id} at ({x}, {y}).")

            # Get node details from user via console input
            while True: name = input(f" > Enter node name (e.g., Entrance, Room 101) [Leave blank for none]: ").strip(); break # Add validation if needed
            while True: node_type = input(f" > Enter node type (e.g., Room, Intersection, Elevator, Doorway) [Leave blank for generic]: ").strip(); break
            while True: landmark = input(f" > Enter nearby landmark (optional, e.g., 'the water fountain') [Leave blank for none]: ").strip(); break

            # Calculate real coordinates if scale is set
            real_x, real_y = x, y # Default to pixel coords
            if scale_pixels_per_meter and scale_pixels_per_meter > 0:
                real_x = x / scale_pixels_per_meter
                real_y = y / scale_pixels_per_meter # Assuming origin is top-left
                print(f"   Scaled coordinates: ({real_x:.2f}m, {real_y:.2f}m)")

            node_data = {
                "id": node_id,
                "name": name if name else f"Point {node_counter}", # Provide a default name if blank
                "type": node_type if node_type else "PointOfInterest",
                "landmark": landmark if landmark else None,
                "pixel_x": x,   # Store original pixel coords for finding nearest
                "pixel_y": y,
                "x": real_x,    # Store potentially scaled coords for map.json
                "y": real_y,
                # Add other fields if needed (e.g., accessibility info)
            }
            # Remove landmark if it's None or empty for cleaner JSON
            if not node_data["landmark"]: del node_data["landmark"]

            nodes.append(node_data)
            node_counter += 1
            redraw_display()
            print(f"Node {node_id} added. Current mode: {mode}. Press 'L' for Link mode, 'S' for Scale mode.")

        # === LINK MODE: Select start node ===
        elif mode == 'LINK_START':
            clicked_node = find_nearest_node(x, y)
            if clicked_node:
                link_start_node = clicked_node
                print(f"Link start node selected: {link_start_node['id']} ({link_start_node['name']})")
                mode = 'LINK_END'
                redraw_display() # Highlight the selected node
                print(f"Current mode: {mode}. Click on the destination node.")
            else:
                print("No node found near click. Click closer to a node to start a link.")

        # === LINK MODE: Select end node and create link ===
        elif mode == 'LINK_END':
            clicked_node = find_nearest_node(x, y)
            if clicked_node and link_start_node:
                if clicked_node['id'] == link_start_node['id']:
                    print("Cannot link a node to itself. Click a different node.")
                else:
                    link_end_node = clicked_node
                    print(f"Link end node selected: {link_end_node['id']} ({link_end_node['name']})")

                    # Check if link (or reverse) already exists
                    link_exists = False
                    for lnk in links:
                        if (lnk['startNode'] == link_start_node['id'] and lnk['endNode'] == link_end_node['id']) or \
                           (lnk['startNode'] == link_end_node['id'] and lnk['endNode'] == link_start_node['id']):
                            link_exists = True
                            break

                    if link_exists:
                        print(f"Link between {link_start_node['id']} and {link_end_node['id']} already exists.")
                    else:
                        # Add the link (consider adding both directions if graph is undirected)
                        links.append({"startNode": link_start_node['id'], "endNode": link_end_node['id']})
                        # links.append({"startNode": link_end_node['id'], "endNode": link_start_node['id']}) # Uncomment for bidirectional
                        print(f"Link created: {link_start_node['id']} <-> {link_end_node['id']}")
                        redraw_display()

                    # Reset for next link
                    link_start_node = None
                    mode = 'LINK_START' # Go back to selecting the start of the *next* link
                    print(f"\nLink added. Current mode: {mode}. Click start node for next link, or press 'N' for Node mode.")

            elif not clicked_node:
                print("No node found near click. Click closer to the destination node.")
            else: # link_start_node was somehow None
                print("Error: Link start node was lost. Resetting.")
                link_start_node = None
                mode = 'LINK_START'
                redraw_display()


        # === SCALE MODE: Select first point ===
        elif mode == 'SCALE_START':
             scale_points = [(x, y)]
             print(f"Scale: First point selected at ({x},{y}).")
             mode = 'SCALE_END'
             redraw_display()
             print(f"Current mode: {mode}. Click the second point for scaling.")

        # === SCALE MODE: Select second point and calculate scale ===
        elif mode == 'SCALE_END':
             if not scale_points: # Should not happen
                 mode = 'SCALE_START'
                 return
             scale_points.append((x, y))
             print(f"Scale: Second point selected at ({x},{y}).")
             pixel_distance = calculate_distance(scale_points[0], scale_points[1])
             print(f"   Pixel distance between points: {pixel_distance:.2f} pixels")

             if pixel_distance < 1:
                 print("Scale points are too close. Please try again.")
                 scale_points = []
                 mode = 'SCALE_START'
                 redraw_display()
                 return

             while True:
                 try:
                     real_distance_str = input(" > Enter the real-world distance between these points in METERS: ")
                     real_distance_meters = float(real_distance_str)
                     if real_distance_meters <= 0:
                         print("Distance must be positive.")
                     else:
                         break
                 except ValueError:
                     print("Invalid input. Please enter a number (e.g., 5.5).")

             scale_pixels_per_meter = pixel_distance / real_distance_meters
             print(f"--- Scale Set: {scale_pixels_per_meter:.2f} pixels per meter ---")

             # Optionally update existing nodes' real coordinates based on new scale
             confirm_update = input(" > Update existing node coordinates with this new scale? (y/N): ").lower()
             if confirm_update == 'y':
                 print("   Updating existing node coordinates...")
                 for node in nodes:
                     node['x'] = node['pixel_x'] / scale_pixels_per_meter
                     node['y'] = node['pixel_y'] / scale_pixels_per_meter
                 print("   Existing node coordinates updated.")

             scale_points = [] # Clear scale points visually
             mode = 'NODE'     # Return to node mode
             redraw_display()
             print(f"Current mode: {mode}. Continue adding nodes or press 'L' for Link mode.")


def save_map(filepath="map.json"):
    """Builds the final map structure and saves it to a JSON file."""
    print(f"\nSaving map data to {filepath}...")
    if not nodes:
        print("WARN: No nodes were defined. Saving an empty map.")

    map_data = {
        "nodes": [],
        "links": [] # Store simplified links here if needed, or rely on graph builder
    }

    # Prepare nodes for JSON (using potentially scaled 'x', 'y')
    for node in nodes:
        # Create a copy to avoid modifying the original node dict if needed later
        json_node = node.copy()
        # Remove temporary pixel coordinates used by the tool
        del json_node['pixel_x']
        del json_node['pixel_y']
        map_data["nodes"].append(json_node)


    # Prepare links and calculate weights (important!)
    final_links = []
    links_added = set() # To handle potential bidirectional duplicates
    for link in links:
        start_id = link['startNode']
        end_id = link['endNode']

        # Ensure we don't add duplicates if tool added both directions
        if (start_id, end_id) in links_added or (end_id, start_id) in links_added:
            continue

        node1 = get_node_by_id(start_id)
        node2 = get_node_by_id(end_id)

        if node1 and node2:
            distance = get_real_distance(node1, node2)
            # Add link for graph builder - our NodeMapManager handles weights usually
            # Or you can add weight here directly if needed by other tools
            final_links.append({
                "startNode": start_id,
                "endNode": end_id,
                # "weight": distance # Uncomment if you want weight explicitly in JSON
            })
            links_added.add((start_id, end_id))
            # If the NodeMapManager build logic *doesn't* assume bidirectionality
            # from the "links" array, add the reverse link explicitly here.
            # final_links.append({
            #     "startNode": end_id,
            #     "endNode": start_id,
            # })
            # links_added.add((end_id, start_id))

        else:
            print(f"WARN: Could not find nodes for link {start_id} <-> {end_id}. Skipping link.")

    map_data["links"] = final_links

    # Write the JSON file
    try:
        with open(filepath, 'w') as f:
            json.dump(map_data, f, indent=2)
        print(f"Map successfully saved to {filepath}")
    except Exception as e:
        print(f"ERROR: Could not save map file: {e}")

# --- Main Execution ---
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Semi-automated floor plan map generator.")
    parser.add_argument("image_path", help="Path to the floor plan image file.")
    parser.add_argument("-o", "--output", default="map.json", help="Output path for the map.json file.")
    args = parser.parse_args()

    if not os.path.exists(args.image_path):
        print(f"ERROR: Image file not found at '{args.image_path}'")
        exit(1)

    image = cv2.imread(args.image_path)
    if image is None:
        print(f"ERROR: Could not load image from '{args.image_path}'. Check file format and permissions.")
        exit(1)

    display_image = image.copy()

    # --- Instructions ---
    print("\n--- Floor Plan Map Generator ---")
    print("Instructions:")
    print(" - Mode 'NODE' (default): Click to add a navigation node (room, intersection, etc.).")
    print(" - Mode 'LINK': Click two existing nodes to create a path between them.")
    print(" - Mode 'SCALE': Click two points, then enter the real distance between them in meters.")
    print("Controls:")
    print(" - Left Click: Perform action based on current mode.")
    print(" - 'N': Switch to NODE mode.")
    print(" - 'L': Switch to LINK mode (select start node).")
    print(" - 'S': Switch to SCALE mode (select first point).")
    print(" - 'W': Write current nodes and links to map.json file.")
    print(" - 'Q': Quit the tool (prompts to save).")
    print("---------------------------------")
    print(f"Current Mode: {mode}")

    # --- OpenCV Window Setup ---
    cv2.namedWindow("Floor Plan Map Generator")
    cv2.setMouseCallback("Floor Plan Map Generator", mouse_callback)
    redraw_display() # Initial display

    # --- Interaction Loop ---
    while True:
        key = cv2.waitKey(1) & 0xFF # Wait indefinitely until a key is pressed

        if key == ord('q'): # Quit
            confirm = input("Save before quitting? (Y/n): ").lower()
            if confirm != 'n':
                 save_map(args.output)
            break
        elif key == ord('n'): # Switch to Node mode
             if mode != 'NODE':
                 print("\nSwitched to NODE mode. Click to add nodes.")
                 mode = 'NODE'
                 link_start_node = None # Clear any pending link start
                 scale_points = [] # Clear any pending scale points
                 redraw_display() # Update highlight/markers if any
        elif key == ord('l'): # Switch to Link mode
             if mode != 'LINK_START' and mode != 'LINK_END':
                 if not nodes:
                     print("Add some nodes first before creating links!")
                     continue
                 print("\nSwitched to LINK mode. Click the starting node for the link.")
                 mode = 'LINK_START'
                 link_start_node = None # Clear any previous selection
                 scale_points = []
                 redraw_display()
        elif key == ord('s'): # Switch to Scale mode
             if mode != 'SCALE_START' and mode != 'SCALE_END':
                 print("\nSwitched to SCALE mode. Click the first point for scaling.")
                 mode = 'SCALE_START'
                 link_start_node = None
                 scale_points = []
                 redraw_display()
        elif key == ord('w'): # Write (Save)
             save_map(args.output)
             # Stay in current mode after saving


    cv2.destroyAllWindows()
    print("Map generator closed.")

usage: ipykernel_launcher.py [-h] [-o OUTPUT] image_path
ipykernel_launcher.py: error: the following arguments are required: image_path


SystemExit: 2

c:\Users\rohit\perceptaai\cap\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
